# **YOLO ile Görüntü Analiz İşlemleri**

Bölüm 1- Resim Üzerinden Nesne Tespiti ve Sınıflandırma:

Bölüm 2- Video Üzerinden Nesne Tespiti ve Sınıflandırma:

Bölüm 3- Proje Adı: YOLO11s ve ByteTrack ile Trafik Yoğunluğu Ölçümü Projesi

Bölüm 4- Dört Yol Bölgesinin Ayrı Ayrı Trafik Yoğunluğu Analizi



---

♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒

---


# **Bölüm 1- Resim Üzerinden Nesne Tespiti ve Sınıflandırma:**



---


**1. Aşama — Google Drive’ı Colab’a Bağlama**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')



---


2.Aşama - Google Drive’da YOLO_Projeleri
klasörünü ve bu klasör içine 1_Resim_Tespit adlı bir klasör oluşturur.

DİKKAT: BU AŞAMAYI SADECE UYGULAMAYA İLK BAŞLADIĞINDA ÇALIŞTIRMALISIN.

KALDIĞIN YERDEN DEVAM EDECEKSEN DOSYAYI YÜKLEYİP DOSYA-KAYDET YAPABİLİRSİN.

ÖNEMLİ:

parents=True, eksik olan üst klasörleri de oluşturur. Bu kod YOLO_Projeleri klasörünü ve onun içinde 1_Resim_Tespit klasörünü hazırlar. Klasörler zaten varsa exist_ok=True hata oluşmasını önler.

In [ ]:
from pathlib import Path

klasor = Path("/content/drive/MyDrive/YOLO_Projeleri/1_Resim_Tespit")
klasor.mkdir(parents=True, exist_ok=True)

print("Klasör hazır:", klasor)



---


3. Aşama - Colab’daki Yolo_agi_nesne_tanima.ipynb dosyasının bir kopyasını 1_Resim_Tespit klasörüne koyar. Orijinal dosya yerinde kalır.

DİKKAT: BU AŞAMAYI SADECE UYGULAMAYA İLK BAŞLADIĞINDA ÇALIŞTIRMALISIN.

- !cp: Colab’a dosya kopyalama komutu çalıştırır.
- İlk tırnak içindeki yol: Kopyalanacak not defterinin adresi.
- İkinci tırnak içindeki yol: Kopyanın konacağı klasör.

In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/Yolo_agi_nesne_tanima.ipynb" "/content/drive/MyDrive/YOLO_Projeleri/1_Resim_Tespit"



---


4. Aşama — Ultralytics Kütüphanesini Kurma

*Bu kod, YOLO modelini kullanmamızı sağlayan Ultralytics kütüphanesini kurar. -U, kütüphane zaten kuruluysa günceller.*

“Bu aşamada, YOLO modellerini kullanmamızı sağlayan Ultralytics kütüphanesini Colab ortamına kuruyoruz. Kurulum çıktısında görünen sürüm numarası, YOLO modeline değil Ultralytics paketine aittir. Henüz bir model seçmedik; modeli sonraki aşamada yükleyeceğiz.”

In [ ]:
!pip install -U ultralytics



---


5. Aşama — YOLO Modelini Tanıma ve Yükleme

Bu uygulamada önceden eğitilmiş YOLO11 Nano modelini kullanacağız. Model, görüntüde hangi nesnelerin bulunduğunu ve nerede olduklarını tahmin eder.

COCO ile YOLO arasındaki fark nedir?

| Kavram   | Ne işe yarar?                                                                  |
| -------- | ------------------------------------------------------------------------------ |
| **COCO** | Modelin öğrenmesinde kullanılan görüntü ve etiketlerden oluşan veri kümesidir. |
| **YOLO** | Görüntülerde nesneleri tespit eden model ailesidir.                            |


Etiketler, örneğin bir fotoğraftaki nesnenin “otomobil” olduğunu ve görüntünün neresinde bulunduğunu belirtir. Model bu örneklerden öğrenir.

Kullanacağımız model neleri tespit edebilir?

***- COCO ile eğitilmiş bu model; insan, otomobil, otobüs, kedi, köpek ve sandalye gibi 80 nesne sınıfını tespit etmek üzere hazırlanmıştır.

***- yolo11n.pt adındaki n, Nano sürümünü belirtir. Küçük ve hızlı olduğu için başlangıç uygulamamızda bunu kullanıyoruz.

***- Modeli yüklerken COCO görüntülerini değil, modelin önceden öğrendiği bilgileri taşıyan ağırlıkları yüklüyoruz.

***- Model her nesneyi tanımaz. Örneğin vida, bu 80 sınıf arasında değildir.

***- Bildiği nesnelerde de hata yapabilir. Bulanıklık, yetersiz ışık ve nesnelerin birbirini kapatması tespiti zorlaştırabilir.

*** -YOLO’nun COCO üzerinde eğitilmiş standart nesne tespit modeli 80 sınıfı tanımak üzere hazırlanmıştır. Bazı örnekler:

| Grup               | Örnek sınıflar                                |
| ------------------ | --------------------------------------------- |
| İnsan ve ulaşım    | İnsan, bisiklet, otomobil, motosiklet, otobüs |
| Hayvanlar          | Kedi, köpek, kuş, at                          |
| Ev eşyaları        | Sandalye, kanepe, yatak, yemek masası         |
| Mutfak eşyaları    | Şişe, bardak, çatal, kaşık                    |
| Elektronik eşyalar | Televizyon, dizüstü bilgisayar, fare, klavye  |


| Veri kümesi          | İçeriği                                                    | Kullanım alanı                                         |
| -------------------- | ---------------------------------------------------------- | ------------------------------------------------------ |
| **Objects365**       | 365 farklı nesne kategorisi                                | Çok çeşitli günlük nesneleri tespit etme               |
| **Open Images**      | Çok çeşitli nesnelerin bulunduğu geniş görüntü koleksiyonu | Genel nesne tespiti                                    |
| **KITTI**            | Araç, yaya ve bisikletli içeren sürüş görüntüleri          | Trafik ve otonom sürüş                                 |
| **VisDrone**         | Drone ile çekilmiş görüntülerde araçlar ve insanlar        | Yukarıdan görüntüleme, küçük nesne tespiti             |
| **SKU-110K**         | Raflarda yoğun biçimde yerleştirilmiş ürünler              | Mağaza ve raf analizi                                  |
| **Kendi veri kümen** | Senin çekip etiketlediğin görüntüler                       | Vida türü, çizik veya üretim hatası gibi özel görevler |




1.   yolo11n.pt	Nano	En küçük ve hızlı

2.   yolo11s.pt	Small	Biraz daha doğru, daha yavaş

3.   yolo11m.pt	Medium	Daha güçlü, daha ağır

4.   yolo11l.pt	Large	Yüksek hesaplama ihtiyacı

5.   yolo11x.pt	Extra Large	En ağır sürüm















In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/YOLO_Projeleri/1_Resim_Tespit/yolo11s.pt")

#print("YOLO11 Small modeli yüklendi.")



---


6. Aşama — Test Görüntüsünü İndirme ve Gösterme

Burada:

wget: Görseli internetten indirir.

-O test_resmi.jpg: Dosyayı bu adla kaydeder.

display(...): Görseli Colab ekranında gösterir.

width=600: Yalnızca ekrandaki gösterim genişliğidir; görüntü dosyasını değiştirmez.

In [ ]:
from IPython.display import Image, display

!wget -q https://s3img.sm.mncdn.com/mnresize/1170/684/Files/img/Blog/trafik-cezasi/trafik-cezasi-ucretleri-01.jpg -O /content/drive/MyDrive/YOLO_Projeleri/1_Resim_Tespit/test_resmi.jpg

display(Image(filename="test_resmi.jpg", width=400))





---


7. Aşama — Görüntüdeki Nesneleri Tespit Etme

Bu aşamada görüntüyü YOLO modeline veriyoruz. Model, bulduğu nesnelerin çevresine kutular çizer; sınıf adlarını ve güven skorlarını gösterir.

Buradaki:

source: İncelenecek görüntünün dosya yolu

conf=0.25: Güven oranı en az %25 olan tespitleri kabul eder.

sonuclar[0]: İlk görüntünün tespit sonuçları

save(...): Kutuların çizildiği sonucu kaydeder.

display(...): Sonuç görüntüsünü ekranda gösterir.

In [ ]:
sonuclar = model.predict(source="/content/drive/MyDrive/YOLO_Projeleri/1_Resim_Tespit/test_resmi.jpg", conf=0.25 )

sonuclar[0].save(filename="/content/drive/MyDrive/YOLO_Projeleri/1_Resim_Tespit/tespit_sonucu.jpg")

display(Image(filename="/content/drive/MyDrive/YOLO_Projeleri/1_Resim_Tespit/tespit_sonucu.jpg", width=300))

YOLO yalnızca görüntünün üzerine kutu çizmez. Her tespit için nesnenin sınıfını, güven skorunu ve kutunun köşe koordinatlarını sayısal olarak üretir. Bu bilgiler daha sonra nesne sayma, konum belirleme veya AI Agent karar sisteminde kullanılabilir.



---


8. Aşama — Tespit Edilen Nesnelerin Bilgilerini Okuma

YOLO yalnızca kutu çizmez. Her nesnenin sınıfını, güven skorunu ve görüntüdeki konumunu sayısal olarak da verir. Bu kod, bulunan nesneleri sırayla ekrana yazdırır.

In [ ]:
for kutu in sonuclar[0].boxes:

    sinif_id = int(kutu.cls[0])
    sinif_adi = model.names[sinif_id]
    guven = float(kutu.conf[0])

    x1, y1, x2, y2 = map(int, kutu.xyxy[0].tolist())

    print(
        "Nesne:", sinif_adi,
        "| Güven:", round(guven, 2),
        "| Koordinatlar:", x1, y1, x2, y2
    )

Koordinatlar nasıl okunur?

(x1, y1) → Kutunun sol üst köşesi.

(x2, y2) → Kutunun sağ alt köşesi.

| Kod                              | Görevi                                       |
| -------------------------------- | -------------------------------------------- |
| `for kutu in sonuclar[0].boxes:` | Bulunan nesne kutularını sırayla inceler.    |
| `sinif_id`                       | Nesnenin sınıf numarasını alır.              |
| `sinif_adi`                      | Sınıf numarasına karşılık gelen adı bulur.   |
| `guven`                          | Tespitin güven skorunu alır.                 |
| `x1, y1, x2, y2`                 | Kutunun köşe koordinatlarını alır.           |
| `round(guven, 2)`                | Güven skorunu iki ondalık basamağa yuvarlar. |




---



---



---



**Ek Bilgi 1:**

İsteğe Göre Bilgisayardan İstenen Görsel Yüklenerek Nesne Tespiti ve Sınıflandırma Yapılabilir



*   files.upload() → Dosya seçme penceresini açar ve seçtiğimiz
dosyayı yükler.

*   next(iter(yuklenen_dosyalar)) → Yüklenen ilk dosyanın adını alır. Bu aşamada tek görsel seçelim.

*   resim_yolu → Seçilen dosyanın adını tutar.

*   print(...) → Dosyanın adını ekrana yazdırır.












In [ ]:
from google.colab import files

yuklenen_dosyalar = files.upload()

resim_yolu = next(iter(yuklenen_dosyalar))

print("Seçilen görsel:", resim_yolu)

**Ek Bilgi 2:**

İsteğe Göre Arayüz Üzerinden Nesne Tespiti ve Sınıflandırması Yapılabilir.

Örnek: Gradio Arayüz Kütüphanesini Kurma




Gradio, Python ile geliştirilen yapay zekâ modellerine tarayıcı üzerinden kullanılabilen basit bir arayüz ekler. Kullanıcı arayüzden bir görüntü yükler; görüntü YOLO modeline gönderilir ve kutulu sonuç ekranda gösterilir. Gradio nesne tespitini kendisi yapmaz, yalnızca kullanıcı ile YOLO modeli arasında arayüz görevi görür.

In [ ]:
!pip install -U gradio

5.Aşama: Gradio Kütüphanesinin Yüklenmesi ve Sürüm Kontrolü

In [ ]:
import gradio as gr

print("Gradio başarıyla yüklendi.")
print("Gradio sürümü:", gr.__version__)

***- Gradio, makine öğrenmesi ve veri bilimi projeleriniz için hızlıca web tabanlı arayüzler oluşturmanızı sağlayan açık kaynaklı bir Python kütüphanesidir.

***- Bu aşamada, önceden ayrı çalışan kod hücreleri tek bir arayüzde birleştirilerek yüklenen trafik görseli YOLO11 Nano modeline iletilir. Model, görsel üzerindeki nesneleri tespit edip etrafını kutularla işaretler ve bulunan nesne sayılarını Türkçe olarak listeler.

***- Arayüz çalıştırıldığında oluşan geçici bağlantı üzerinden bir görsel seçip Submit/Gönder butonuna basarak sağ panelde tespit sonuçlarını ve nesne sayılarını anında görüntüleyebilirsiniz.

Bu Python kodu, YOLO11 nesne tespiti modelini Gradio web arayüzü ile birleştiren tam bir Gradio uygulama bileşenidir.

In [ ]:
from collections import Counter
import gradio as gr

turkce_adlar = {
    "person": "İnsan",
    "bicycle": "Bisiklet",
    "car": "Otomobil",
    "motorcycle": "Motosiklet",
    "bus": "Otobüs",
    "truck": "Kamyon",
    "traffic light": "Trafik ışığı",
    "cat": "Kedi",
    "dog": "Köpek",
    "bird": "Kuş"
}


def nesneleri_tani(resim_yolu):

    if resim_yolu is None:
        return None, "Lütfen bir görüntü yükleyin."

    sonuc = model.predict(
        source=resim_yolu,
        conf=0.75,
        imgsz=960,
        verbose=False
    )[0]

    # YOLO'nun kutu çizdiği görüntü
    kutulu_goruntu = sonuc.plot()

    # BGR renk düzenini RGB'ye çevir
    kutulu_goruntu = kutulu_goruntu[:, :, ::-1]

    # Bulunan sınıfları say
    sinif_numaralari = sonuc.boxes.cls.int().tolist()
    sinif_adlari = [
        model.names[numara]
        for numara in sinif_numaralari
    ]

    sayim = Counter(sinif_adlari)

    if not sayim:
        ozet = "Nesne tespit edilemedi."
    else:
        satirlar = ["TESPİT EDİLEN NESNELER"]

        for sinif_adi, adet in sayim.items():
            turkce_ad = turkce_adlar.get(sinif_adi, sinif_adi)
            satirlar.append(f"{turkce_ad}: {adet}")

        ozet = "\n".join(satirlar)

    return kutulu_goruntu, ozet


uygulama = gr.Interface(
    fn=nesneleri_tani,

    inputs=gr.Image(
        type="filepath",
        label="Görüntü yükleyin"
    ),

    outputs=[
        gr.Image(label="YOLO nesne tespit sonucu"),
        gr.Textbox(label="Nesne sayımı", lines=10)
    ],

    title="YOLO11 Görüntüde Nesne Tanıma",
    description=(
        "Bir görüntü yükleyin. YOLO11 Nano modeli "
        "görüntüdeki nesneleri bulup kutularla gösterecektir."
    )
)

uygulama.launch(share=True)

## **1. BÖLÜM SONU**

♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒

# **Bölüm 2- Video Üzerinden Nesne Tespiti ve Sınıflandırma:**



---


1.Aşama: Drive'a bağlanma

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


2.Aşama: Video Tespit İşlemleri için Dosya Oluşturma

---


DİKKAT: BU AŞAMAYI SADECE UYGULAMAYA İLK BAŞLADIĞINDA ÇALIŞTIRMALISIN.

KALDIĞIN YERDEN DEVAM EDECEKSEN DOSYAYI YÜKLEYİP ARTIK DOSYA-KAYDET YAPABİLİRSİN.

In [ ]:
from pathlib import Path

klasor = Path("/content/drive/MyDrive/YOLO_Projeleri/2_Video_Tespit")
klasor.mkdir(parents=True, exist_ok=True)

print("Klasör hazır:", klasor)



---


3.Aşama: Colab dosyasını 2_Tespit_Video klasörüne kaydetme.

DİKKAT: BU AŞAMAYI SADECE UYGULAMAYA İLK BAŞLADIĞINDA ÇALIŞTIRMALISIN. KALDIĞIN YERDEN DEVAM EDECEKSEN DOSYAYI YÜKLEYİP RTIK DOSYA-KAYDET AŞAMALARINI KULLANABİLİRSİN.

In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/Yolo_agi_nesne_tanima.ipynb" "/content/drive/MyDrive/YOLO_Projeleri/2_Video_Tespit"



---


4. Aşama — Videoyu Colab’a İndirme

(İnternet üzerinden)

DİKKAT: BU AŞAMAYI SADECE UYGULAMAYA İLK BAŞLADIĞINDA ÇALIŞTIRMALISIN.

Bu aşamada, nesne tespiti yapacağımız örnek trafik videosunu internetten google drive ve colab ortamına indiriyoruz.YOLO videoyu doğrudan tek parça olarak incelemez; videodaki kareleri sırayla işleyerek her karedeki nesneleri ayrı ayrı tespit eder. Video henüz YOLO tarafından işlenmemekte, yalnızca giriş verisi olarak hazırlanmaktadır.

In [ ]:
!wget -O /content/drive/MyDrive/YOLO_Projeleri/2_Video_Tespit/trafik_video.mp4 \
"https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4"

4. Aşama — Videoyu Colab’a İndirme

(Bilgisayara İndirilen Videonun Colab Ortamına Yüklenmesi)

DİKKAT: BU AŞAMAYI SADECE UYGULAMAYA İLK BAŞLADIĞINDA ÇALIŞTIRMALISIN.

Aşağıda ücretsiz trafik videoları var. Ancak Colab üzerinden bağlanmaya izin vermez. İstersen bilgisayarına indirip bilgisayardan yükleme aşaması olan 4.Aşama / A Maddesi aşamasına geçebilirsin.

https://pixabay.com/tr/videos/kent-trafik-otoban-215258/

https://www.pexels.com/search/videos/

In [ ]:
from google.colab import files
from IPython.display import Video, display

# Bilgisayardan videoyu yükle
yuklenen = files.upload()
video_adi = next(iter(yuklenen))

# Yüklenen videoyu oynat
display(Video(video_adi, embed=True, width=500))

Bu aşamada GPU ya geçiş yapılabilir eğer geçiş yapılırsa drive bağlantısı tekrar yapılır. Geçiş yapıldığında Ultralytics kütüphanesinin ve yolo modelinin tekrar yüklenmesi gerekmektedir.

In [ ]:
!pip install -U ultralytics

5.Aşama : Yolo modeli indirme

DİKKAT: Colab oturumu sıfırlanınca bellekteki model değişkeni silinir. Yöntem .pt dosyasını Drive’da saklamak olsa bile her yeni oturumda dosyayı belleğe yüklemek gerekir.

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/YOLO_Projeleri/2_Video_Tespit/yolo11s.pt")

print("YOLO11 Small modeli yüklendi.")

6.aşama: İndirilen Videoyu Kontrol:

In [ ]:
import os

video_yolu = "/content/drive/MyDrive/YOLO_Projeleri/2_Video_Tespit/trafik_video.mp4"

if os.path.exists(video_yolu):
    print("Video var mı? Evet")
    print("Video boyutu:",
          round(os.path.getsize(video_yolu) / 1024 / 1024, 2),
          "MB")
    print("Video yolu:", video_yolu)
else:
    print("Video bulunamadı. Dosya yolunu kontrol et.")

7. Aşama — Videoyu Colab Ekranında Oynatma

In [ ]:
from IPython.display import Video, display

display(Video(
    "/content/drive/MyDrive/YOLO_Projeleri/2_Video_Tespit/trafik_video.mp4",
    embed=True,
    width=500
))



---


8. Aşama — Videodaki Nesneleri Tespit Etme

Bu aşamada YOLO, videodaki kareleri sırayla işler. Her karede nesnelerin sınıfını, konumunu ve güven skorunu tahmin eder. Kutular işlenmiş karelerin üzerine çizilerek yeni bir sonuç videosu oluşturulur. Bu işlem henüz nesne takibi değildir; aynı araç farklı karelerde yeniden tespit edilir.

Bu aşamada YOLO, videonun karelerini sırayla inceleyerek nesneleri bulur. Nesnelerin çevresine kutular çizilir ve sonuç yeni bir video olarak kaydedilir.

Önceki dersteki model yükleme hücresinin çalıştırılmış olması gerekir.

for döngüsü sonuçları sırayla alarak videonun işlenmesini yürütür. pass, her sonuç için ayrıca bir işlem yapmadığımızı belirtir; YOLO tespiti ve kaydı gerçekleştirir.

In [ ]:
video_sonuclari = model.predict(
    source="/content/drive/MyDrive/YOLO_Projeleri/2_Video_Tespit/trafik_video.mp4",
    conf=0.25,
    imgsz=640,
    save=True,
    stream=True,
    project="/content/drive/MyDrive/YOLO_Projeleri/2_Video_Tespit/",
    name="tespit",
    exist_ok=True,
    verbose=False
)

# stream=True sonucu kare kare üretir.
# Döngü, bütün video tamamlanıncaya kadar çalışır.
for _ in video_sonuclari:
    pass

print("Video nesne tespiti tamamlandı.")



---


9. Aşama — Sonuç Videosunu Oynatılabilir Biçime Dönüştürme

YOLO’nun kaydettiği videoyu, Colab ekranında oynatılabilmesi için H.264 kodlamalı MP4 biçimine dönüştürüyoruz.

In [ ]:
# Doğru kaynak yolunu belirterek dönüştürme işlemi
!ffmpeg -y -i "/content/drive/MyDrive/YOLO_Projeleri/2_Video_Tespit/tespit/trafik_video.avi" -vcodec libx264 -acodec aac "/content/drive/MyDrive/YOLO_Projeleri/2_Video_Tespit/tespit/traffic_2_donusturulmus.mp4"

#print("Dönüştürme tamamlandı!")



---


10. Aşama — Nesne Tespiti Yapılmış Videoyu İzleme

In [ ]:
from IPython.display import Video, display

display(
    Video(
        "/content/drive/MyDrive/YOLO_Projeleri/2_Video_Tespit/tespit/traffic_2_donusturulmus.mp4",
        embed=True,
        width=600
    )
)



---


11. Aşama — Sonuç Videosunu Bilgisayara İndirme

In [ ]:
from google.colab import files

files.download("/content/drive/MyDrive/YOLO_Projeleri/2_Video_Tespit/tespit/traffic_2_donusturulmus.mp4")

**DİKKAT:**

GPU yalnızca yoğun model eğitimi ve video işleme sırasında kullanılmalıdır. GPU gerektirmeyen kod yazma, veri inceleme ve açıklama aşamalarında CPU ortamına dönmek, Colab kaynaklarının gereksiz tüketilmesini önler.

CPU ortamına geçtiğini doğrulamak için

In [ ]:
import torch

print("CUDA kullanılabilir mi?", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Aktif ortam: GPU")
    print("GPU modeli:", torch.cuda.get_device_name(0))
else:
    print("Aktif ortam: CPU")
    print("GPU bağlantısı bulunmuyor.")

♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒



---



---



---



# **Bölüm 3 - Proje Adı: YOLO11s ve ByteTrack ile Trafik Yoğunluğu Ölçümü Projesi**



---


**Aşama 1 — Projenin amacı ve genel işlem akışı**

Proje Bilgi:

  Bu projede, sabit kamera ile kaydedilmiş bir trafik videosundaki araçların ve yol yoğunluğunun belirlenmesi amaçlanmıştır. COCO veri kümesiyle önceden eğitilmiş YOLO11s modeli kullanılarak otomobil, motosiklet, otobüs ve kamyonlar tespit edilmiş; ByteTrack algoritmasıyla araçlar video boyunca takip edilmiştir.

  Kamera görüntüsündeki yollar ayrı bölgelere ayrılmış ve araç kutularının her yol üzerinde kapladığı alan hesaplanmıştır. Elde edilen doluluk oranlarına göre yollar düşük yoğunlukta yeşil, orta yoğunlukta sarı ve yüksek yoğunlukta kırmızı renkle gösterilmiştir. Sonuç videosunda araç kutuları, takip numaraları, yol doluluk oranları ve yoğunluk seviyeleri birlikte sunulmuştur.

Bu uygulama gelecekteki trafik yoğunluğunu tahmin etmemekte, videoda görülen mevcut trafik yoğunluğunu ölçmektedir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')



---


Aşama 2 — GPU ortamının kontrol edilmesi


Trafik videosu çok sayıda görüntü karesinden oluştuğu için nesne tespiti işlemleri GPU üzerinde gerçekleştirilecektir. Bu aşamada PyTorch kullanılarak Colab ortamında CUDA destekli NVIDIA GPU bulunup bulunmadığı kontrol edilmiştir. GPU, YOLO modelinin matris hesaplamalarını CPU’ya göre daha hızlı gerçekleştirir.

Bu aşamada Google Colab ortamında GPU’nun kullanılabilir olup olmadığı kontrol edilmiştir. CPU bilgisayarın genel işlemlerini yürütürken GPU, çok sayıda görüntü ve matematiksel işlemi paralel olarak gerçekleştirebilir. Bu nedenle YOLO modelinin videodaki yüzlerce kareyi analiz etmesi GPU üzerinde CPU’ya göre daha hızlıdır.

CPU; videoyu okuma, verileri düzenleme ve sonuç videosunu oluşturma işlemlerinde kullanılır. GPU ise YOLO11s modelinin araçları tespit etme hesaplamalarını hızlandırır. Colab ortamında NVIDIA Tesla T4 GPU etkinleştirilerek nesne tespit işlemleri bu donanım üzerinde gerçekleştirilmiştir.

In [ ]:
import torch

gpu_var_mi = torch.cuda.is_available()

print("CUDA kullanılabilir mi?", gpu_var_mi)

if gpu_var_mi:
    print("GPU modeli:", torch.cuda.get_device_name(0))
    print("Kullanılacak cihaz: cuda:0")
else:
    print("GPU bulunamadı.")
    print("Kullanılacak cihaz: cpu")



---
EĞER GPU ORTAMINA GEÇİŞ YAPILMADIYSA:

Aşama 3 - T4 GPU’nun etkinleştirilmesi

BU AŞAMAYI AŞAĞIDA VERİLEN MANÜ BASAMAKLARINI KULLANARAK YAPABİLİRSİN VEYA AŞAĞIDAKİ KODU ÇALIŞTIRIP GEÇİŞİ YAPABİLİRSİN.
Colab menüsünden:

Çalışma zamanı menüsünü aç.

Çalışma zamanı türünü değiştir seçeneğine bas.

Donanım hızlandırıcı bölümünden T4 GPU seç.

Kaydet düğmesine bas.

Colab yeniden bağlanana kadar bekle.


Önemli: Çalışma zamanı değiştiğinde /content klasörü sıfırlanabilir. Ancak henüz yeni trafik projesinin dosyalarını hazırlamadığımız için şu anda sorun oluşturmaz.

In [ ]:
import torch

gpu_var_mi = torch.cuda.is_available()

print("CUDA kullanılabilir mi?", gpu_var_mi)

if gpu_var_mi:
    print("GPU modeli:", torch.cuda.get_device_name(0))
    print("Kullanılacak cihaz: cuda:0")
else:
    print("GPU bulunamadı.")



---


4.Aşama - Google Drive’da YOLO_Projeleri
klasörünün içine 3_Proje_Tespit adlı bir klasör oluşturur.

DİKKAT: BU AŞAMAYI SADECE UYGULAMAYA İLK BAŞLADIĞINDA ÇALIŞTIRMALISIN.

KALDIĞIN YERDEN DEVAM EDECEKSEN DOSYAYI YÜKLEYİP DOSYA-KAYDET YAPABİLİRSİN.

In [ ]:
from pathlib import Path

klasor = Path("/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit")
klasor.mkdir(parents=True, exist_ok=True)

print("Klasör hazır:", klasor)



---


5. Aşama - Colab’daki Yolo_agi_nesne_tanima.ipynb dosyasının bir kopyasını 1_Resim_Tespit klasörüne koyar. Orijinal dosya yerinde kalır.

DİKKAT: BU AŞAMAYI SADECE UYGULAMAYA İLK BAŞLADIĞINDA ÇALIŞTIRMALISIN.

- !cp: Colab’a dosya kopyalama komutu çalıştırır.
- İlk tırnak içindeki yol: Kopyalanacak not defterinin adresi.
- İkinci tırnak içindeki yol: Kopyanın konacağı klasör.

In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/Yolo_agi_nesne_tanima.ipynb" "/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit"



---


   6.Aşama - Ultralytics kütüphanesinin kurulması

Bu aşamada YOLO modellerini Python üzerinden kullanmayı sağlayan Ultralytics kütüphanesi kurulmuştur. Daha sonra COCO veri kümesinin 80 nesne sınıfıyla önceden eğitilmiş YOLO11s modeli yüklenmiştir. Model sıfırdan eğitilmemekte; otomobil, motosiklet, otobüs ve kamyon gibi daha önce öğrendiği sınıflar kullanılmaktadır.

In [ ]:
!pip install -U ultralytics



---


7.Aşama - YOLO11s modelinin yüklenmesi

Aşağıdaki kod COCO ile eğitilmiş YOLO11 Small modelini indirir ve belleğe yükler.

In [ ]:
from ultralytics import YOLO
import ultralytics

model = YOLO("/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/yolo11s.pt")

print("Ultralytics sürümü:", ultralytics.__version__)
print("Yüklenen model: YOLO11 Small")
print("Model dosyası: yolo11s.pt")



---


8.Aşama - Trafik videosunun Colab’a indirilmesi

Bu aşamada sabit kamerayla çekilmiş bir kavşak videosu Colab ortamına indirilmiştir. Video analizinden önce çözünürlük, saniyedeki kare sayısı, toplam kare sayısı ve video süresi belirlenmiştir. YOLO modeli videoyu tek parça olarak değil, görüntü karelerini sırayla işleyerek analiz edecektir.

In [ ]:
!wget -O /content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/trafik_akisi.mp4 \
"https://raw.githubusercontent.com/antonmilev/TrafficDetection/main/vid.mp4"

Buradaki:

wget: Dosyayı internetten indirir.

-O: İndirilen dosyanın adını ve konumunu belirler.



---


9.Aşama - Video özelliklerinin incelenmesi

In [ ]:
import cv2
import os

video_yolu = "/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/trafik_akisi.mp4"

video = cv2.VideoCapture(video_yolu)

fps = video.get(cv2.CAP_PROP_FPS)
kare_sayisi = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
genislik = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
yukseklik = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
sure = kare_sayisi / fps

video.release()

print("Video bulundu mu?", os.path.exists(video_yolu))
print("Çözünürlük:", genislik, "x", yukseklik)
print("FPS:", fps)
print("Toplam kare:", kare_sayisi)
print("Video süresi:", round(sure, 2), "saniye")

Çözünürlük	Her karenin piksel boyutu

FPS	Bir saniyedeki görüntü karesi

Kare sayısı	YOLO’nun inceleyeceği toplam görüntü sayısı

Süre	Kare sayısı ÷ FPS

Video gerçekte 10 saniye sürer ve YOLO yaklaşık 250 ayrı görüntüyü inceleyecektir.



---


10.Aşama - İndirilen Trafik Videosunun Oynatılması

In [ ]:
from IPython.display import Video, display

display(
    Video(
        "/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/trafik_akisi.mp4",
        embed=True,
        width=600
    )
)



---


11.Aşama - Videodan "jpg" formatında Örnek Kare Çıkarılması

Bu aşamada trafik videosunun orta bölümünden bir görüntü karesi alınmıştır. YOLO11s modeli yalnızca trafikle ilişkili otomobil, motosiklet, otobüs ve kamyon sınıflarını arayacak şekilde çalıştırılmıştır. Tek kare testiyle modelin video üzerindeki nesne tespit performansı, bütün video işlenmeden önce kontrol edilmiştir.

In [ ]:
import cv2

video = cv2.VideoCapture("/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/trafik_akisi.mp4")

toplam_kare = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
orta_kare_numarasi = toplam_kare // 2

video.set(cv2.CAP_PROP_POS_FRAMES, orta_kare_numarasi)

basarili, ornek_kare = video.read()

video.release()

if basarili:
    ornek_kare_yolu = "/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/ornek_kare.jpg"
    cv2.imwrite(ornek_kare_yolu, ornek_kare)

    print("Örnek kare çıkarıldı.")
    print("Kare numarası:", orta_kare_numarasi)
    print("Dosya yolu:", ornek_kare_yolu)
else:
    print("Video karesi okunamadı.")



---


12.Aşama - Örnek kareden alınan görselin (jpg) gösterilmesi

In [ ]:
from IPython.display import Image, display

display(
    Image(
        filename="/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/ornek_kare.jpg",
        width=600
    )
)




---


13.Aşama - Yolo ile Videodan Alınan "jpg" Görselden Araç Tespiti Yapılması

In [ ]:
ornek_sonuc = model.predict(
    source="/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/ornek_kare.jpg",
    classes=[2, 3, 5, 7],
    conf=0.25,
    imgsz=960,
    device=0,
    quantize=16,
    verbose=False
)[0]

ornek_sonuc.save(
    filename="/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/ornek_kare_sonuc.jpg"
)

print("Tek kare araç tespiti tamamlandı.")



---


14.Aşama - Yolo ile "jpg" üzerinden tespit edilen araç sınıflarının ve sayılarının gösterilmesi

In [ ]:
display(
    Image(
        filename="/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/ornek_kare_sonuc.jpg",
        width=600
    )
)



---


15.Aşama — Örnek Karede Tespit Edilen Araçların Sayılması

Bu aşamada YOLO11s modelinin örnek görüntü karesinde tespit ettiği araçların sınıf numaraları alınmıştır. Sınıf numaraları otomobil, motosiklet, otobüs ve kamyon adlarına dönüştürülmüş; Counter kullanılarak her araç türünün sayısı hesaplanmıştır. Örnek karede 24 otomobil ve 1 otobüs olmak üzere toplam 25 motorlu araç tespit edilmiştir.

In [ ]:
from collections import Counter
from pathlib import Path

# Görselin adı ve yolu
dosya_yolu = ornek_sonuc.path
print("Dosya adı:", Path(dosya_yolu).name)
print("Dosya yolu:", dosya_yolu)

sinif_numaralari = ornek_sonuc.boxes.cls.int().tolist()
sinif_adlari = [model.names[numara] for numara in sinif_numaralari]
sayim = Counter(sinif_adlari)

turkce_adlar = {
    "car": "Otomobil",
    "motorcycle": "Motosiklet",
    "bus": "Otobüs",
    "truck": "Kamyon"
}

print("\nÖRNEK KARE ARAÇ SAYIMI")

for sinif_adi, adet in sayim.items():
    print(turkce_adlar.get(sinif_adi, sinif_adi), ":", adet)

print("Toplam motorlu araç:", sum(sayim.values()))



---


16. Aşama — ByteTrack ile Araçların Video Boyunca Takip Edilmesi

YOLO her karede araçları bulur fakat aynı otomobilin önceki karedeki otomobil olduğunu tek başına bilmez.

ByteTrack, araçlara kimlik numarası vererek aynı aracı ardışık karelerde takip eder.

Bu aşamada YOLO11s tarafından tespit edilen araçlar ByteTrack algoritmasıyla video boyunca takip edilmiştir. Takip sistemi her araca bir kimlik numarası atayarak aynı aracın ardışık görüntü karelerinde eşleştirilmesini sağlar. Her karedeki aktif araç sayısı, araç kutuları ve takip kimlikleri daha sonraki trafik yoğunluğu hesabında kullanılmak üzere kaydedilmiştir.

In [ ]:
import time

baslangic = time.perf_counter()

benzersiz_idler = set()
kare_kayitlari = []

takip_sonuclari = model.track(
    source=video_yolu,
    tracker="bytetrack.yaml",
    persist=True,
    classes=[2, 3, 5, 7],
    conf=0.25,
    imgsz=960,
    device=0,
    quantize=16,
    stream=True,
    save=True,
    project="/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/",
    name="takip_sonucu",
    exist_ok=True,
    verbose=False
)

for kare_numarasi, sonuc in enumerate(takip_sonuclari):

    if sonuc.boxes.id is not None:

        takip_idleri = (
            sonuc.boxes.id
            .int()
            .cpu()
            .tolist()
        )

        siniflar = (
            sonuc.boxes.cls
            .int()
            .cpu()
            .tolist()
        )

        kutular = (
            sonuc.boxes.xyxy
            .cpu()
            .tolist()
        )

        benzersiz_idler.update(takip_idleri)

    else:
        takip_idleri = []
        siniflar = []
        kutular = []

    kare_kayitlari.append({
        "kare": kare_numarasi,
        "aktif_arac": len(takip_idleri),
        "takip_idleri": takip_idleri,
        "siniflar": siniflar,
        "kutular": kutular
    })

bitis = time.perf_counter()

print("Araç takibi tamamlandı.")
print("İşlenen kare sayısı:", len(kare_kayitlari))
print("Üretilen farklı takip kimliği:", len(benzersiz_idler))
print("Toplam işlem süresi:", round(bitis - baslangic, 2), "saniye")
print("Sonuç klasörü:/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/takip_sonucu")



---


17.Aşama — Araç Takip Sonuç Videosunun Bulunması

Bu aşamada YOLO11s ve ByteTrack tarafından oluşturulan takip videosu çıktı klasöründe bulunmuştur. Ultralytics tarafından üretilen AVI videosu, Colab tarayıcısıyla uyumlu olması için H.264 kodlamalı MP4 biçimine dönüştürülmüş ve ekranda oynatılmıştır. Videodaki takip kimlikleri, araçların ardışık karelerde aynı nesne olarak eşleştirilmesini göstermektedir.

In [ ]:
from pathlib import Path

sonuc_klasoru = Path(
    "/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/takip_sonucu"
)

video_dosyalari = (
    list(sonuc_klasoru.glob("*.avi")) +
    list(sonuc_klasoru.glob("*.mp4")) +
    list(sonuc_klasoru.glob("*.mov"))
)

print("Bulunan video dosyaları:")

for dosya in video_dosyalari:
    print(dosya)

takip_video_yolu = str(video_dosyalari[0])

print("Kullanılacak video:", takip_video_yolu)



---


18.Aşama — Videoyu H.264 MP4 biçimine dönüştürme

In [ ]:
!ffmpeg -y -loglevel error \
-i "{takip_video_yolu}" \
-vcodec libx264 \
-pix_fmt yuv420p \
-an \
"/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/takip_sonucu/trafik_takip_sonucu.mp4"


print("Video dönüşümü tamamlandı.")

Buradaki:

libx264: H.264 video kodlaması yapar.

yuv420p: Tarayıcılarla uyumlu renk biçimi kullanır.

-an: Ses kanalını kaldırır.

-y: Aynı isimli dosya varsa yeniden oluşturur.

Bu dönüşüm, YOLO tespitlerini değiştirmez. Yalnızca videonun sıkıştırma biçimini değiştirir.



---


19.Aşama — Takip videosunu oynatma

In [ ]:
from IPython.display import Video, display

display(
    Video(
        "/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/takip_sonucu/trafik_takip_sonucu.mp4",
        embed=True,
        width=700
    )
)



---


20.Aşama — Yoğunluk Hesaplama Yönteminin Belirlenmesi ve Yol Seçimine Hazırlık

UNUTMA: Kimlik numarası değişmiyorsa takip başarılıdır.

Yolun üzerine yarı saydam bir renk katmanı ekleriz: Bu katmanı  renklerle sembolize ederiz.

Yeşil: Trafik düşük

Sarı: Trafik orta

Kırmızı: Trafik yüksek

## Yoğunluk nasıl hesaplanacak?

Yalnızca araç sayısını bilmek yeterli değildir. Araçların yolun ne kadarını kapladığını hesaplayacağız.

$$
\text{Doluluk oranı (\%)} =
\frac{\text{Yol üzerinde araçların kapladığı alan}}
{\text{Toplam yol alanı}}
\times 100
$$

### Örnek hesaplama

- Yol alanı: **400.000 piksel**
- Araçların kapladığı alan: **120.000 piksel**

$$
\frac{120.000}{400.000} \times 100 = \%30
$$

Bu örnekte araçlar, yol alanının **%30’unu** kaplıyor.




---
Dikkat:

Örnek Olarak Öncelikle Trafik Takibi(Predict) Yaptığımız Videodan ("trafik_takip_sonucu.mp4") Bir Kare Alıp  ("jpg") O Kare Üzerinde Seçim Yapıyoruz.

20. Aşama 1.Bölüm: Videonun ilk karesini al, kaydet ve göster

In [ ]:
import cv2
import matplotlib.pyplot as plt

video_yolu = "/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/takip_sonucu/trafik_takip_sonucu.mp4"
kare_yolu = "/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/takip_sonucu/ornek_kare.jpg"

video = cv2.VideoCapture(video_yolu)
basarili, kare = video.read()
video.release()

if not basarili or kare is None:
    raise RuntimeError(
        f"Videodan kare okunamadı: {video_yolu}"
    )

# Alınan kareyi resim dosyası olarak kaydet
kaydedildi = cv2.imwrite(kare_yolu, kare)

if not kaydedildi:
    raise RuntimeError("Örnek kare kaydedilemedi.")

print("Örnek kare kaydedildi:", kare_yolu)

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(kare, cv2.COLOR_BGR2RGB))
plt.title("Videodan alınan ilk kare")
plt.axis("off")
plt.show()



---


21. Aşama — Ana Yatay Yol Bölgesinin Oluşturulması

(Kaydedilen jpg kareyi oku ve yol bölgesini çiz)



Bu aşamada trafik yoğunluğunun ölçüleceği ana yol bölgesi bir çokgenle tanımlanmıştır. Yol bölgesi görüntü boyutuna oranlı koordinatlarla seçilmiş ve seçimin doğruluğunu kontrol etmek amacıyla yarı saydam yeşil renkle gösterilmiştir. Görüntünün yol dışındaki bölümleri yoğunluk hesabına alınmayacaktır.

Dikkat: Bu kod, görüntünün orta kısmında dört köşeli bir bölge tanımlıyor.

Görüntülerde koordinat başlangıcı sol üst köşedir:

(0, 0). x sağa, y aşağıya doğru artar.

Her köşe [x, y] şeklinde yazılır.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# İlk hücrede kaydedilen görüntüyü oku
kare_yolu = "/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/takip_sonucu/ornek_kare.jpg"
kare = cv2.imread(kare_yolu)

if kare is None:
    raise FileNotFoundError(
        "Örnek kare okunamadı. Önce birinci hücreyi çalıştır."
    )

yukseklik, genislik = kare.shape[:2]
print("Görüntü boyutu:", genislik, "x", yukseklik)

# Yol bölgesinin dört köşesi: [x, y]
yol_cokgeni = np.array([
    [0, int(yukseklik * 0.43)],             # Sol üst
    [genislik - 1, int(yukseklik * 0.46)],  # Sağ üst
    [genislik - 1, int(yukseklik * 0.66)],  # Sağ alt
    [0, int(yukseklik * 0.56)]              # Sol alt
], dtype=np.int32)

print("Yol köşe koordinatları:")
print(yol_cokgeni)

# Görüntünün kopyası üzerine sınır çiz
gosterim = kare.copy()

cv2.polylines(
    gosterim,
    [yol_cokgeni],
    isClosed=True,
    color=(0, 255, 0),
    thickness=3
)

plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(gosterim, cv2.COLOR_BGR2RGB))
plt.title("Seçilen yol bölgesi")
plt.axis("off")
plt.show()




---


Aşama 21 — Seçilen Yol Bölgesinin Yarı Saydam Yeşil Renklendirilmesi

1. Görüntünün kopyasını oluşturur

           Boyamayı bu kopyada yapar; asıl kare değişmez.

2. Yol bölgesinin içini yeşille doldurur

           yol_cokgeni içindeki köşelerin çevrelediği alanı tamamen yeşile boyar. Bu aşamada boyanan bölgenin altındaki yol görünmez.

3. Boyanmış görüntüyü orijinaliyle karıştırır.

           Sonuç = boyalı görüntü × 0,30 + orijinal görüntü × 0,70 yapılır.

           Böylece yol bölgesinde %30 yeşil katman ve %70 asıl görüntü birleşir.

DİKKAT:

“Orijinal %100 + yeşil %30” yaparsak ağırlıklar toplamı %130 olur; bu, normal bir saydam katman karışımı olmaz. Şeffaf yeşil katman kullanmanın anlamı, yeşilin bir kısmının görüntünün önüne geçmesi ve orijinal görüntünün o oranda azalmasıdır.

In [ ]:
from pathlib import Path

renkli_katman = kare.copy()

yesil = (0, 255, 0)  # OpenCV: BGR

cv2.fillPoly(
    renkli_katman,
    [yol_cokgeni],
    yesil
)

renkli_kare = cv2.addWeighted(
    renkli_katman,
    0.30,
    kare,
    0.70,
    0
)

# Yol sınırını daha belirgin çiz
cv2.polylines(
    renkli_kare,
    [yol_cokgeni],
    isClosed=True,
    color=yesil,
    thickness=4
)

# Kaydedilecek dosyanın konumu
kayit_yolu = "/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/takip_sonucu/yol_bolgesi.jpg"

# Klasör yoksa oluştur
Path(kayit_yolu).parent.mkdir(parents=True, exist_ok=True)

# Görüntüyü kaydet ve sonucu kontrol et
if cv2.imwrite(kayit_yolu, renkli_kare):
    print("Görüntü kaydedildi:", kayit_yolu)
else:
    raise RuntimeError("Görüntü kaydedilemedi.")



---


22.Aşama — Seçilen Yol Bölgesinin Görüntülenmesi ve Kontrol Edilmesi


Yeşil alan:

Ana yatay yolu kapsamalı,

Üstteki binaları kapsamamalı,

Alttaki ağaçlık alanı kapsamamalı,

Yol üzerinde hareket eden araçların bulunduğu bölgeyi içine almalıdır.

Bu aşamada yeşil renk “trafik düşük” anlamına gelmiyor. Yalnızca seçilen yol bölgesini kontrol ediyoruz.

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Gösterilecek görüntünün konumu
dosya_yolu = "/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/takip_sonucu/yol_bolgesi.jpg"

renkli_kare = cv2.imread(dosya_yolu)

if renkli_kare is None:
    raise FileNotFoundError(
        f"Görüntü okunamadı: {dosya_yolu}"
    )

renkli_kare_rgb = cv2.cvtColor(
    renkli_kare,
    cv2.COLOR_BGR2RGB
)

print("Gösterilen dosya:", dosya_yolu)

plt.figure(figsize=(14, 8))
plt.imshow(renkli_kare_rgb)
plt.title("Trafik yoğunluğu ölçülecek ana yol bölgesi")
plt.axis("off")
plt.show()



---


23.Aşama — Araçların Yol Üzerinde Kapladığı Alanın Hesaplanması


### Doluluk yüzdesi hesabı:

$$
\text{Doluluk yüzdesi} =
\frac{\text{Araç kutularının yol içindeki birleşik alanı}}
{\text{Yol bölgesinin alanı}}
\times 100
$$



---


24. Aşama — Yol Maskesinin Oluşturulması

Bu kod, `yol_cokgeni` ile seçilen bölgenin piksel alanını hesaplar. Seçilen bölge yolun tamamını kapsayacak şekilde belirlenmelidir.

1. Görüntünün veya video karesinin boyutunu alır.
2. Görüntüyle aynı boyutta, tamamen siyah bir maske oluşturur.
3. Seçilen çokgenin içini maskede beyaza boyar.
4. Beyaz pikselleri sayarak seçilen bölgenin alanını bulur.

| Maske değeri | Renk | Anlamı |
|---|---|---|
| `0` | Siyah | Seçilen bölgenin dışında |
| `255` | Beyaz | Seçilen yol bölgesinin içinde |

Bu maskede `255` değerindeki pikseller seçilen yol alanını, `0` değerindeki pikseller ise bu alanın dışını gösterir. Sayılan beyaz pikseller, doluluk oranı hesabında kullanılacak **toplam yol alanıdır**.

**Dikkat:** Bu kod araçları bulmaz ve araçların kapladığı alanı hesaplamaz. Yalnızca seçilen yol bölgesinin alanını hesaplar. Araçların yol içinde kapladığı alan, sonraki aşamada araç tespit kutuları kullanılarak ayrıca hesaplanır.

DİKKAT: Bu kod tek başına araçları veya trafik yoğunluğunu ölçmez; yalnızca yol alanını hesaplar.

In [ ]:
import cv2
import numpy as np

kaynak_yolu = (
    "/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/"
    "takip_sonucu/yol_bolgesi.jpg"
)

# JPG görselini aç
kare = cv2.imread(kaynak_yolu)

if kare is None:
    raise FileNotFoundError(f"Görsel bulunamadı: {kaynak_yolu}")

yukseklik, genislik = kare.shape[:2]

# Seçili yol alanını maske üzerinde beyaza boya
yol_maskesi = np.zeros((yukseklik, genislik), dtype=np.uint8)
cv2.fillPoly(yol_maskesi, [yol_cokgeni], 255)

# Beyaz pikselleri, yani seçili poligonun alanını say
yol_alani = cv2.countNonZero(yol_maskesi)

print("Kullanılan görsel:", kaynak_yolu)
print("Seçilen yol bölgesinin alanı:", yol_alani, "piksel")



---


25. Aşama — Her Karede Yol İçindeki Araç Alanını Hesaplama

Bu kod, kare_kayitlari içindeki her video karesini sırayla inceler. Otomobil, motosiklet, otobüs ve kamyon kutularını araç maskesine çizer. Ardından bu maskeyi yol maskesiyle kesiştirerek yalnızca seçilen yol bölgesinde kalan araç kutusu piksellerini sayar.

In [ ]:
import cv2
import numpy as np
import pandas as pd

# Yalnızca motorlu araç sınıflarını say
arac_siniflari = {"car", "motorcycle", "bus", "truck"}

yukseklik, genislik = yol_maskesi.shape

if yol_alani == 0:
    raise ValueError("Yol alanı sıfır. yol_cokgeni ve yol_maskesi kontrol edilmeli.")

kare_sonuclari = []
doluluk_oranlari = []

for sira, kayit in enumerate(kare_kayitlari):

    # Bu kare için boş araç maskesi
    arac_maskesi = np.zeros(
        (yukseklik, genislik),
        dtype=np.uint8
    )

    # Her araç kutusunu maskeye beyaz olarak çiz
    for kutu, sinif_id in zip(kayit["kutular"], kayit["siniflar"]):

        sinif_adi = model.names[int(sinif_id)]

        if sinif_adi not in arac_siniflari:
            continue

        x1, y1, x2, y2 = map(int, kutu)

        # Kutuyu görüntü sınırlarının içinde tut
        x1 = max(0, min(x1, genislik - 1))
        x2 = max(0, min(x2, genislik - 1))
        y1 = max(0, min(y1, yukseklik - 1))
        y2 = max(0, min(y2, yukseklik - 1))

        if x2 > x1 and y2 > y1:
            cv2.rectangle(
                arac_maskesi,
                (x1, y1),
                (x2, y2),
                255,
                thickness=-1
            )

    # Araç kutularının yalnızca seçili yol içinde kalan bölümünü al
    yol_icindeki_arac_maskesi = cv2.bitwise_and(
        arac_maskesi,
        yol_maskesi
    )

    # Üst üste gelen kutuların pikselleri bir kez sayılır
    arac_piksel_alani = cv2.countNonZero(
        yol_icindeki_arac_maskesi
    )

    doluluk = arac_piksel_alani / yol_alani * 100
    doluluk_oranlari.append(doluluk)

    kare_sonuclari.append({
        "kare": kayit.get("kare", sira),
        "aktif_arac": kayit["aktif_arac"],
        "yol_icindeki_arac_kutu_alani_piksel": arac_piksel_alani,
        "doluluk_yuzdesi": doluluk
    })

sonuc_tablosu = pd.DataFrame(kare_sonuclari)

print("Hesaplanan kare sayısı:", len(sonuc_tablosu))
print("Yol alanı:", yol_alani, "piksel")
print("\nİlk 5 kare sonucu:")
display(sonuc_tablosu.head())

print(
    f"\nOrtalama yol doluluğu: "
    f"%{sonuc_tablosu['doluluk_yuzdesi'].mean():.2f}"
)



---


26. Aşama - Video Üzerinde Yol İçi Araç Alanının Gösterilmesi

Bu kod, videoyu kare kare işler ve seçtiğimiz yol bölgesindeki araç kutularının kapladığı alanı görselleştirir.
- Yeşil alan: Seçilen yol bölgesini gösterir.
- Kırmızı alan: Araç kutularının yol bölgesinin içinde kalan kısmını gösterir.
- Bilgi paneli: O karede yol içinde kalan araç kutularının piksel alanını ve yol doluluk yüzdesini yazar.

In [ ]:
import cv2
import numpy as np
import subprocess
from pathlib import Path
from IPython.display import Video, display

# Sadece motorlu araç sınıfları
arac_siniflari = {"car", "motorcycle", "bus", "truck"}

# Yol alanı maskesi ile videonun çözünürlüğü eşleşmeli
yukseklik, genislik = yol_maskesi.shape

giris_video = cv2.VideoCapture(video_yolu)

if not giris_video.isOpened():
    raise RuntimeError("Video açılamadı. video_yolu adresini kontrol edin.")

fps = giris_video.get(cv2.CAP_PROP_FPS)
video_genislik = int(giris_video.get(cv2.CAP_PROP_FRAME_WIDTH))
video_yukseklik = int(giris_video.get(cv2.CAP_PROP_FRAME_HEIGHT))

if (video_genislik, video_yukseklik) != (genislik, yukseklik):
    giris_video.release()
    raise ValueError(
        f"Boyutlar eşleşmiyor. Video: {video_genislik}×{video_yukseklik}, "
        f"yol maskesi: {genislik}×{yukseklik}"
    )

# Sonuçları Drive klasörüne kaydet
hedef_klasor = Path(
    "/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/takip_sonucu"
)
hedef_klasor.mkdir(parents=True, exist_ok=True)

gecici_video = str(hedef_klasor / "yol_arac_piksel_gecici.mp4")
sonuc_video = str(hedef_klasor / "yol_arac_piksel.mp4")

video_yazici = cv2.VideoWriter(
    gecici_video,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (genislik, yukseklik)
)

if not video_yazici.isOpened():
    giris_video.release()
    raise RuntimeError("Sonuç videosu oluşturulamadı.")

yol_ici = yol_maskesi > 0
yesil = np.array([0, 255, 0], dtype=np.float32)  # BGR
kirmizi = np.array([0, 0, 255], dtype=np.float32)  # BGR

islenen_kare = 0

for kare_numarasi, kayit in enumerate(kare_kayitlari):

    basarili, video_karesi = giris_video.read()

    if not basarili:
        print("Video kareleri sona erdi.")
        break

    # O karedeki araç kutularını bir maskede birleştir
    arac_maskesi = np.zeros((yukseklik, genislik), dtype=np.uint8)

    for kutu, sinif_id in zip(kayit["kutular"], kayit["siniflar"]):
        sinif_adi = model.names[int(sinif_id)]

        if sinif_adi not in arac_siniflari:
            continue

        x1, y1, x2, y2 = map(int, kutu)

        x1 = max(0, min(x1, genislik - 1))
        x2 = max(0, min(x2, genislik - 1))
        y1 = max(0, min(y1, yukseklik - 1))
        y2 = max(0, min(y2, yukseklik - 1))

        if x2 > x1 and y2 > y1:
            cv2.rectangle(
                arac_maskesi,
                (x1, y1),
                (x2, y2),
                255,
                thickness=-1
            )

    # Araç kutularının yalnızca yol içinde kalan bölümünü bul
    yol_icindeki_araclar = cv2.bitwise_and(
        arac_maskesi,
        yol_maskesi
    )

    arac_piksel_alani = cv2.countNonZero(yol_icindeki_araclar)
    doluluk = arac_piksel_alani / yol_alani * 100

    # Yol alanını yarı saydam yeşil göster
    sonuc_kare = video_karesi.copy()
    sonuc_kare[yol_ici] = (
        sonuc_kare[yol_ici].astype(np.float32) * 0.72
        + yesil * 0.28
    ).astype(np.uint8)

    # Yol içindeki araç kutusu piksellerini kırmızı göster
    arac_bolge = yol_icindeki_araclar > 0
    sonuc_kare[arac_bolge] = (
        sonuc_kare[arac_bolge].astype(np.float32) * 0.25
        + kirmizi * 0.75
    ).astype(np.uint8)

    # Seçili yol sınırını çiz
    yol_cokgeni_goruntu = np.array(
        yol_cokgeni,
        dtype=np.int32
    ).reshape(-1, 1, 2)

    cv2.polylines(
        sonuc_kare,
        [yol_cokgeni_goruntu],
        True,
        (0, 255, 0),
        4
    )

    # Bilgi paneli
    cv2.rectangle(sonuc_kare, (20, 20), (650, 115), (0, 0, 0), -1)
    cv2.putText(
        sonuc_kare,
        f"Yol icindeki arac kutusu: {arac_piksel_alani} piksel",
        (35, 55),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.75,
        (255, 255, 255),
        2
    )
    cv2.putText(
        sonuc_kare,
        f"Yol dolulugu: %{doluluk:.2f}",
        (35, 95),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.75,
        (255, 255, 255),
        2
    )

    video_yazici.write(sonuc_kare)
    islenen_kare += 1

giris_video.release()
video_yazici.release()

# MP4'ü tarayıcıda daha uyumlu oynatılacak biçime çevir
subprocess.run(
    [
        "ffmpeg", "-y", "-loglevel", "error",
        "-i", gecici_video,
        "-vcodec", "libx264",
        "-pix_fmt", "yuv420p",
        sonuc_video
    ],
    check=True
)

print("İşlenen kare sayısı:", islenen_kare)
print("Sonuç videosu:", sonuc_video)

display(Video(sonuc_video, embed=True, width=900))



---


Aşama 27 — Doluluk Değerlerinin Tabloya Eklenmesi ve Hareketli Ortalamanın Hesaplanması

Bu aşamada her kareye ait doluluk yüzdesi ve araç sayısı bir tabloya eklenir. Video FPS değeri kullanılarak kare numaraları yaklaşık zamana dönüştürülür. Yaklaşık bir saniyelik kayıt penceresindeki doluluk değerlerinin ortalaması alınarak kısa süreli iniş çıkışlar yumuşatılır. Bu işlem ani değişimleri silmez; grafikteki dalgalanmaları daha anlaşılır hâle getirir.

Bu aşamada her kare için hesaplanan yol doluluk yüzdeleri veri tablosuna eklenmiş ve ani değişimleri azaltmak için bir saniyelik hareketli ortalama hesaplanmıştır.

In [ ]:
import cv2
import pandas as pd

# Gerekli önceki sonuçları kontrol et
if "kare_kayitlari" not in globals():
    raise NameError(
        "kare_kayitlari bulunamadı. "
        "Önce Aşama 5'teki araç takip kodunu çalıştırın."
    )

if "doluluk_oranlari" not in globals():
    raise NameError(
        "doluluk_oranlari bulunamadı. "
        "Önce Aşama 9.2'deki doluluk hesaplama kodunu çalıştırın."
    )

# Kayıt sayılarının eşitliğini kontrol et
if len(kare_kayitlari) != len(doluluk_oranlari):
    raise ValueError(
        "Kare kayıtlarının sayısıyla doluluk değerlerinin "
        "sayısı eşit değil."
    )

# Veri tablosunu oluştur
grafik_verisi = pd.DataFrame({
    "kare": [
        kayit["kare"]
        for kayit in kare_kayitlari
    ],

    "aktif_arac": [
        kayit["aktif_arac"]
        for kayit in kare_kayitlari
    ]
})

# FPS değerini videodan yeniden oku
video = cv2.VideoCapture(video_yolu)
fps = video.get(cv2.CAP_PROP_FPS)
video.release()

# Kare numarasını zamana dönüştür
grafik_verisi["zaman_saniye"] = (
    grafik_verisi["kare"] / fps
)

# Bir saniyeye karşılık gelen kare sayısı
pencere_boyutu = max(
    1,
    int(round(fps))
)

print("FPS:", fps)
print(
    "Hareketli ortalama pencere boyutu:",
    pencere_boyutu,
    "kare"
)

# Doluluk sonuçlarını tabloya ekle
grafik_verisi["doluluk_yuzdesi"] = (
    doluluk_oranlari
)

# Bir saniyelik hareketli ortalama
grafik_verisi["doluluk_ortalamasi"] = (
    grafik_verisi["doluluk_yuzdesi"]
    .rolling(
        window=pencere_boyutu,
        center=True,
        min_periods=1
    )
    .mean()
)

print(
    grafik_verisi[
        [
            "zaman_saniye",
            "aktif_arac",
            "doluluk_yuzdesi",
            "doluluk_ortalamasi"
        ]
    ].head()
)

Burada 27 araç görüntünün tamamında takip ediliyor. Ancak doluluk hesabına yalnızca seçtiğimiz yatay yol bölgesi içindeki kutu parçaları giriyor. Bu nedenle aktif araç sayısı yüksekken yol doluluğu yaklaşık %3,5 olabilir.


doluluk_ortalamasi ilk karelerde daha düşük görünebilir; hareketli ortalama video başlangıcında daha az komşu kareye sahiptir.




---


Aşama 28 — Yolun En Boş, En Yoğun ve Ortalama Doluluk Değerlerinin Hesaplanması


Şimdi minimum, maksimum ve ortalamayı çıkarmamız gerekiyor.
Minimum, maksimum ve ortalama değerleri videodaki trafik durumunu tek bir sonuçla özetlemek için çıkarıyoruz:

Minimum doluluk: Yolun en boş olduğu anı gösterir.

Maksimum doluluk: Yolun en yoğun olduğu anı gösterir.

Ortalama doluluk: Video boyunca genel trafik seviyesini gösterir.

| İşlem | Anlamı |
|---|---|
| `.min()` | Sütundaki en düşük doluluk |
| `.max()` | Sütundaki en yüksek doluluk |
| `.mean()` | Sütundaki değerlerin aritmetik ortalaması |

In [ ]:
en_dusuk_doluluk = (
    grafik_verisi["doluluk_ortalamasi"].min()
)

en_yuksek_doluluk = (
    grafik_verisi["doluluk_ortalamasi"].max()
)

ortalama_doluluk = (
    grafik_verisi["doluluk_ortalamasi"].mean()
)

print(
    "En düşük doluluk:",
    round(en_dusuk_doluluk, 2),
    "%"
)

print(
    "En yüksek doluluk:",
    round(en_yuksek_doluluk, 2),
    "%"
)

print(
    "Ortalama doluluk:",
    round(ortalama_doluluk, 2),
    "%"
)

Bu değerlere göre:

Yeşil eşik  → düşük yoğunluk

Sarı eşik   → orta yoğunluk

Kırmızı eşik → yüksek yoğunluk

sınırlarını belirleyeceğiz.



---


29.Aşama — Trafik Yoğunluğu Eşiklerinin Belirlenmesi


Bu kodun mantığı, ölçülen doluluk değerlerini küçükten büyüğe sıralayıp yaklaşık üç gruba ayıracak sınırları bulmaktır. Sınırlar, videomuzdaki verilere göre belirlenir.

| Doluluk oranı | Renk |
|---|---|
| %4 ve altı | Yeşil |
| %4’ten büyük, %5 ve altı | Sarı |
| %5’ten büyük | Kırmızı |

In [ ]:
# Eğitim amacıyla belirlediğimiz sabit doluluk eşikleri
alt_esik = 4.0
ust_esik = 5.0

print("Yeşil–sarı sınırı:", alt_esik, "%")
print("Sarı–kırmızı sınırı:", ust_esik, "%")



---


30. Aşama — Doluluk Değerlerinin Trafik Seviyesine Dönüştürülmesi

Bu aşama eşikleri yeniden hesaplamaz. Önceden hesaplanan alt_esik ve ust_esik değerlerini kullanarak doluluğu sınıflandıran bir fonksiyon oluşturur:

Alt eşiğe kadar → DÜŞÜK

İki eşik arasında → ORTA

Üst eşikten sonra → YÜKSEK

In [ ]:
def yogunluk_sinifi(doluluk):

    if doluluk <= alt_esik:
        return "DÜŞÜK"

    elif doluluk <= ust_esik:
        return "ORTA"

    else:
        return "YÜKSEK"

Örneğin:

| Doluluk | Sonuç |
|---|---|
| %2,55 | DÜŞÜK |
| %3,00 | DÜŞÜK |
| %3,56 | ORTA |
| %4,00 | ORTA |
| %5,14 | YÜKSEK |



---


31. Aşama — Trafik Yoğunluğu Sınıflarının Tabloya Eklenmesi

In [ ]:
grafik_verisi["yogunluk"] = (
    grafik_verisi["doluluk_ortalamasi"]
    .apply(yogunluk_sinifi)
)

print(
    grafik_verisi[
        [
            "zaman_saniye",
            "aktif_arac",
            "doluluk_ortalamasi",
            "yogunluk"
        ]
    ].head(10)
)



---


32. Aşama — Yolun yoğunluğa göre renklendirildiği videonun oluşturulması

(Trafik Yoğunluğu Bilgilerinin Renkli Video Üzerinde Gösterilmesi)

Aşamanın amacı

Orijinal videoyu kare kare okuyacağız. Her karenin hesaplanan yoğunluk sınıfına göre yol bölgesini:

Düşük → Yeşil

Orta → Sarı

Yüksek → Kırmızı

renkle kaplayacağız.

Ayrıca araç kutuları, takip kimlikleri, doluluk oranı ve aktif araç sayısı gösterilecek.

Colab bilgi notu

Bu aşamada her video karesinin daha önce hesaplanan yoğunluk sınıfı okunmuştur. Ana yol bölgesi düşük yoğunlukta yeşil, orta yoğunlukta sarı ve yüksek yoğunlukta kırmızı renkle yarı saydam olarak kaplanmıştır. Araçların sınırlayıcı kutuları ve ByteTrack kimlikleri yol kaplamasının üzerine çizilmiş; doluluk oranı ve aktif araç sayısı bilgi panelinde gösterilmiştir.

In [ ]:
import cv2

giris_video = cv2.VideoCapture(video_yolu)

video_fps = giriş_fps = giris_video.get(
    cv2.CAP_PROP_FPS
)

video_genislik = int(
    giris_video.get(cv2.CAP_PROP_FRAME_WIDTH)
)

video_yukseklik = int(
    giris_video.get(cv2.CAP_PROP_FRAME_HEIGHT)
)

gecici_video_yolu = (
    "/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/takip_sonucu/trafik_yogunluk_gecici.mp4"
)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

video_yazici = cv2.VideoWriter(
    gecici_video_yolu,
    fourcc,
    video_fps,
    (video_genislik, video_yukseklik)
)

renk_haritasi = {
    "DÜŞÜK": (0, 255, 0),     # Yeşil
    "ORTA": (0, 255, 255),    # Sarı
    "YÜKSEK": (0, 0, 255)     # Kırmızı
}

metin_haritasi = {
    "DÜŞÜK": "DUSUK",
    "ORTA": "ORTA",
    "YÜKSEK": "YUKSEK"
}

kare_numarasi = 0

while True:

    basarili, video_karesi = giris_video.read()

    if not basarili:
        break

    if kare_numarasi >= len(grafik_verisi):
        break

    tablo_satiri = grafik_verisi.iloc[
        kare_numarasi
    ]

    yogunluk = tablo_satiri["yogunluk"]

    doluluk = float(
        tablo_satiri["doluluk_ortalamasi"]
    )

    aktif_arac = int(
        tablo_satiri["aktif_arac"]
    )

    yol_rengi = renk_haritasi[yogunluk]

    # 1. Yol bölgesine renk katmanı ekle
    renkli_katman = video_karesi.copy()

    cv2.fillPoly(
        renkli_katman,
        [yol_cokgeni],
        yol_rengi
    )

    video_karesi = cv2.addWeighted(
        renkli_katman,
        0.28,
        video_karesi,
        0.72,
        0
    )

    # 2. Yol bölgesinin dış sınırını çiz
    cv2.polylines(
        video_karesi,
        [yol_cokgeni],
        isClosed=True,
        color=yol_rengi,
        thickness=5
    )

    # 3. Bu karedeki takip sonuçlarını al
    kayit = kare_kayitlari[kare_numarasi]

    kutular = kayit["kutular"]
    takip_idleri = kayit["takip_idleri"]
    siniflar = kayit["siniflar"]

    # 4. Araç kutularını ve kimliklerini çiz
    for kutu, takip_id, sinif_id in zip(
        kutular,
        takip_idleri,
        siniflar
    ):

        x1, y1, x2, y2 = map(int, kutu)

        cv2.rectangle(
            video_karesi,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            thickness=3
        )

        sinif_adi = model.names[sinif_id]

        etiket = (
            f"{sinif_adi} ID:{takip_id}"
        )

        cv2.rectangle(
            video_karesi,
            (x1, max(0, y1 - 30)),
            (x1 + 190, y1),
            (0, 0, 0),
            thickness=-1
        )

        cv2.putText(
            video_karesi,
            etiket,
            (x1 + 5, max(20, y1 - 7)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.60,
            (255, 255, 255),
            thickness=2
        )

    # 5. Siyah yarı saydam bilgi paneli
    panel = video_karesi.copy()

    cv2.rectangle(
        panel,
        (20, 20),
        (520, 175),
        (0, 0, 0),
        thickness=-1
    )

    video_karesi = cv2.addWeighted(
        panel,
        0.60,
        video_karesi,
        0.40,
        0
    )

    # 6. Yoğunluk bilgilerini yaz
    cv2.putText(
        video_karesi,
        "TRAFIK YOGUNLUGU: "
        + metin_haritasi[yogunluk],
        (40, 70),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        yol_rengi,
        thickness=3
    )

    cv2.putText(
        video_karesi,
        f"Yol doluluk: %{doluluk:.2f}",
        (40, 115),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.85,
        (255, 255, 255),
        thickness=2
    )

    cv2.putText(
        video_karesi,
        f"Aktif arac: {aktif_arac}",
        (40, 155),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.85,
        (255, 255, 255),
        thickness=2
    )

    video_yazici.write(video_karesi)

    kare_numarasi += 1

giris_video.release()
video_yazici.release()

print("Renkli yoğunluk videosu oluşturuldu.")
print("İşlenen kare sayısı:", kare_numarasi)
print("Geçici video:", gecici_video_yolu)



---


32. Aşama — Sonuç Videosunun Tarayıcıda Açılabilecek H.264 Biçimine Dönüştürülmesi

| Dosya | Kapsayıcı | Video kodlaması |
|---|---|---|
| `trafik_yogunluk_gecici.mp4` | MP4 | MPEG-4 Part 2 (`mp4v`) |
| `trafik_yogunluk_sonucu.mp4` | MP4 | H.264 (`libx264`) |

DİKKAT: SADECE AVI UZANTIYI MP4'E DEĞİL MP4'Ü DE UYUMLU HALE GETİRMEK İÇİN ÇEVİRİYORUZ.

Amaç, videonun Colab içindeki tarayıcı oynatıcısıyla daha uyumlu olmasını sağlamak.

In [ ]:
!ffmpeg -y -loglevel error \
-i "/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/takip_sonucu/trafik_yogunluk_gecici.mp4" \
-vcodec libx264 \
-pix_fmt yuv420p \
-an \
"/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/takip_sonucu/trafik_yogunluk_sonucu.mp4"

print("H.264 dönüşümü tamamlandı.")



---


33. Aşama — Hazırlanan Trafik Yoğunluğu Videosunun Görüntülenmesi

In [ ]:
from IPython.display import Video, display

display(
    Video(
        "/content/drive/MyDrive/YOLO_Projeleri/3_Proje_Tespit/takip_sonucu/trafik_yogunluk_sonucu.mp4",
        embed=True,
        width=700
    )
)

## **BÖLÜM SONU**



---



---



---



♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒♒


BÖLÜM 4 - Dört Yol Bölgesinin Ayrı Ayrı Trafik Yoğunluğu Analizi
---


1. Aşama — Dört Yol Bölgesinin Ayrı Ayrı Trafik Yoğunluğu Analizi

Bu aşamada YOLO11s araçları tespit eder, ByteTrack ise her araca bir takip numarası vererek video boyunca aynı aracı izler. Görüntüde belirlenen dört yol bölgesinin doluluk oranları ayrı ayrı hesaplanır ve her yol düşük, orta veya yüksek yoğunluk olarak sınıflandırılır. Sonuçlar yeşil, sarı ve kırmızı renklerle videoya işlenir; araç kutuları, takip numaraları ve işlem süreleri gösterilir. Son olarak analiz videosu oluşturularak oynatma ve indirme bağlantısı hazırlanır.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


2.Aşama: Video Tespit İşlemleri için Dosya Oluşturma

---


DİKKAT: BU AŞAMAYI SADECE UYGULAMAYA İLK BAŞLADIĞINDA ÇALIŞTIRMALISIN.

KALDIĞIN YERDEN DEVAM EDECEKSEN DOSYAYI YÜKLEYİP ARTIK DOSYA-KAYDET YAPABİLİRSİN.

Bu aşamada sabit kamerayla çekilmiş bir kavşak videosu Colab ortamına indirilmiştir. Video analizinden önce çözünürlük, saniyedeki kare sayısı, toplam kare sayısı ve video süresi belirlenmiştir. YOLO modeli videoyu tek parça olarak değil, görüntü karelerini sırayla işleyerek analiz edecektir.

In [ ]:
from pathlib import Path

klasor = Path("/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit")
klasor.mkdir(parents=True, exist_ok=True)

print("Klasör hazır:", klasor)



---


3. Aşama - Trafik videosunun Colab’a indirilmesi

Bu aşamada sabit kamerayla çekilmiş bir kavşak videosu Colab ortamına indirilmiştir. Video analizinden önce çözünürlük, saniyedeki kare sayısı, toplam kare sayısı ve video süresi belirlenmiştir. YOLO modeli videoyu tek parça olarak değil, görüntü karelerini sırayla işleyerek analiz edecektir.

In [ ]:
!wget -O /content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/trafik_akisi.mp4 \
"https://raw.githubusercontent.com/antonmilev/TrafficDetection/main/vid.mp4"

Buradaki:

wget: Dosyayı internetten indirir.

-O: İndirilen dosyanın adını ve konumunu belirler.



---


4. Aşama - Colab’daki Yolo_agi_nesne_tanima.ipynb dosyasının bir kopyasını 4_Yol_Video_Tespit klasörüne koyar. Orijinal dosya yerinde kalır.

DİKKAT: BU AŞAMAYI SADECE UYGULAMAYA İLK BAŞLADIĞINDA ÇALIŞTIRMALISIN.

- !cp: Colab’a dosya kopyalama komutu çalıştırır.
- İlk tırnak içindeki yol: Kopyalanacak not defterinin adresi.
- İkinci tırnak içindeki yol: Kopyanın konacağı klasör.

In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/Yolo_agi_nesne_tanima.ipynb" "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/"



---


6. Aşama - Kütüphane Kurulması

In [ ]:
!pip install -U ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/yolo11s.pt")

print("YOLO11 Small modeli yüklendi.")

Uygulamada sabit kamera görüntüsündeki yollar ayrı ilgi bölgeleri olarak tanımlanmıştır. YOLO11s modeli otomobil, motosiklet, otobüs ve kamyonları tespit etmiş; ByteTrack algoritması araçlara takip numarası vermiştir. Her yol bölgesinde araç kutularının kapladığı alan hesaplanarak yol bazlı doluluk yüzdesi elde edilmiştir. Doluluk değerleri videonun kendi dağılımına göre düşük, orta ve yüksek olarak sınıflandırılmış; yollar sırasıyla yeşil, sarı ve kırmızı renklerle gösterilmiştir.

Eleştirel not: Buradaki “doluluk”, gerçek araç yüzey alanı değil, YOLO sınırlayıcı kutularının yol bölgesinde kapladığı alanın yaklaşık oranıdır. Ayrıca eşikler videonun yüzde 33 ve yüzde 66 dilimlerinden üretildiği için sonuçlar mutlak trafik standardı değil, video içindeki göreli yoğunluğu gösterir.



---


7. Aşama — Kaynak Trafik Videosunun İnternetten Colab’a İndirilmesi Bilgisayara İndirilmesi

DİKKAT:

Bu aşamada indirilen videonun 2. saniyesinden bir kare alınır. Görüntü özgün 1920 × 1080 çözünürlükte kaydedilir ve Paint’te işaretlemek üzere bilgisayara indirilir.

In [ ]:
import cv2

from google.colab.patches import cv2_imshow
from google.colab import files


video_yolu = "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/trafik_akisi.mp4"
video = cv2.VideoCapture(video_yolu)

# Videonun 2. saniyesine git
video.set(
    cv2.CAP_PROP_POS_MSEC,
    2000
)

basarili, yol_secim_karesi = video.read()

video.release()


if not basarili:
    raise RuntimeError(
        "Videodan görüntü alınamadı."
    )


resim_yolu = "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/yol_isaretleme_karesi.jpg"

kaydedildi = cv2.imwrite(
    resim_yolu,
    yol_secim_karesi
)


if not kaydedildi:
    raise RuntimeError(
        "Görüntü kaydedilemedi."
    )


print(
    "Görüntü boyutu:",
    yol_secim_karesi.shape[1],
    "x",
    yol_secim_karesi.shape[0]
)

cv2_imshow(yol_secim_karesi)

files.download(resim_yolu)



---


8. Aşama — Alt Yol Sınırının Paint ile İşaretlenmesi

Bilgisayarınız "indirilneler" klasörüne indirilen yol_isaretleme_karesi.jpg görüntüsünü Paint’te açın.

Uygulanacak işlem:

1-Çizgi aracını seçin.

2-Siyah veya kırmızı, belirgin bir renk kullanın.

3-Yalnızca alt yolun görünen asfalt sınırlarını çevreleyin.

4-Ağaç, kaldırım ve çimenlik alanları dışarıda bırakın.

5-Çizgileri birleştirerek kapalı bir çokgen oluşturun.

Görüntüyü yeniden boyutlandırmayın ve kırpmayın.

alt_yol_secili.png adıyla farklı kaydedin.

İçini boyamayın; yalnızca dış sınırı çizmeniz yeterli. İşaretlenmiş görüntüyü buraya yükleyin. Çizgileri 1920 * 1080 koordinatlarına dönüştürerek alt yol maskesini oluşturacağız.

çİZGİLERİN KAYMA SORUNU OLMAMASI İÇİN PAİNTTE DOSYA UZANTISINI PNG OLARAK KAYDET.



---


9. Aşama — Paint Çizgisinden Yol Koordinatlarının Çıkarılması

DİKKAT: ÇALIŞTIRMADAN ÖNCE KODDA YOL İŞARETLEMESİ YAPTIĞIN GÖRSELİN ADERSİNİ BELİRTMEYİ UNUTMA

orijinal_resim_yolu = (
    "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/yol_isaretleme_karesi.jpg"

In [ ]:
import cv2
import numpy as np

from google.colab import files
from google.colab.patches import cv2_imshow


# -------------------------------------------------
# 1. Paint'te işaretlenen görüntüyü yükle
# -------------------------------------------------

import cv2
import shutil
from pathlib import Path
from google.colab import files

# Kaydetmek istediğiniz klasör
# PAİNTTE İŞARETLEDİĞİMİZ GÖRSELİ YÜKLEME KONUMU VE AŞAMASI
hedef_klasor = Path(
    "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit"
)
hedef_klasor.mkdir(parents=True, exist_ok=True)

# Paint'te işaretlediğiniz görüntüyü yükleyin
yuklenen_dosyalar = files.upload()
yuklenen_ad = next(iter(yuklenen_dosyalar))

# Görüntüyü belirlediğiniz Drive klasörüne kopyala
isaretli_dosya_yolu = hedef_klasor / Path(yuklenen_ad).name
shutil.copy2(yuklenen_ad, isaretli_dosya_yolu)

print("İşaretli görüntü kaydedildi:", isaretli_dosya_yolu)

# Analizde kullanılacak görüntüyü kaydedilen konumdan aç
isaretli_gorsel = cv2.imread(str(isaretli_dosya_yolu))

if isaretli_gorsel is None:
    raise RuntimeError("İşaretli görüntü okunamadı.")

# -------------------------------------------------
# 2. Orijinal görüntüyü oku
# Paint’te işaretlediğiniz görselin işaretlenmemiş aslını Drive’dan açar
# -------------------------------------------------

orijinal_resim_yolu = (
    "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/yol_isaretleme_karesi.jpg"
)

orijinal_gorsel = cv2.imread(
    orijinal_resim_yolu
)


if orijinal_gorsel is None:
    raise RuntimeError(
        "Orijinal yol seçim görüntüsü bulunamadı."
    )


if (
    orijinal_gorsel.shape
    != isaretli_gorsel.shape
):
    raise RuntimeError(
        "İki görüntünün boyutları aynı değil. "
        "Paint görüntüsünü kırpmayın veya "
        "yeniden boyutlandırmayın."
    )


# -------------------------------------------------
# 3. Paint'te eklenen çizgiyi bul
# -------------------------------------------------

fark = cv2.absdiff(
    orijinal_gorsel,
    isaretli_gorsel
)

fark_miktari = np.max(
    fark,
    axis=2
)

cizgi_maskesi = np.where(
    fark_miktari > 60,
    255,
    0
).astype(np.uint8)


# Çizgideki küçük boşlukları birleştir
cekirdek = np.ones(
    (7, 7),
    dtype=np.uint8
)

cizgi_maskesi = cv2.morphologyEx(
    cizgi_maskesi,
    cv2.MORPH_CLOSE,
    cekirdek,
    iterations=2
)


# -------------------------------------------------
# 4. En büyük kapalı çizgiyi bul
# -------------------------------------------------

konturlar, _ = cv2.findContours(
    cizgi_maskesi,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)


if not konturlar:
    raise RuntimeError(
        "Paint çizgisi bulunamadı."
    )


en_buyuk_kontur = max(
    konturlar,
    key=cv2.contourArea
)

dis_bukey = cv2.convexHull(
    en_buyuk_kontur
)

cevre = cv2.arcLength(
    dis_bukey,
    True
)


# Dört köşeli çokgeni bul
alt_yol_noktalari = None

for hassasiyet in np.linspace(
    0.005,
    0.10,
    50
):

    yaklasik_cokgen = cv2.approxPolyDP(
        dis_bukey,
        hassasiyet * cevre,
        True
    )

    if len(yaklasik_cokgen) == 4:

        alt_yol_noktalari = (
            yaklasik_cokgen
            .reshape(-1, 2)
            .astype(np.int32)
        )

        break


if alt_yol_noktalari is None:
    raise RuntimeError(
        "Çizgiden dört köşe çıkarılamadı. "
        "Paint çizgisini daha kalın ve kapalı çizin."
    )


# -------------------------------------------------
# 5. Bulunan alanı ön izle
# -------------------------------------------------

katman = orijinal_gorsel.copy()

cv2.fillPoly(
    katman,
    [alt_yol_noktalari],
    (0, 255, 0)
)

onizleme = cv2.addWeighted(
    katman,
    0.30,
    orijinal_gorsel,
    0.70,
    0
)

cv2.polylines(
    onizleme,
    [alt_yol_noktalari],
    True,
    (0, 255, 0),
    5
)


# Ekrana sığdır
kucuk_onizleme = cv2.resize(
    onizleme,
    (960, 540)
)

cv2_imshow(kucuk_onizleme)

# Ön izlemeyi belirlediğiniz Drive klasörüne kaydet
# yEŞİL OLARAK iŞARETLENENGÖRSELİN FOTOGRAFINI DRİVE ORTAMINA KAYDEDER.
onizleme_yolu = hedef_klasor / "yol_bolgesi_onizleme.jpg"

kaydedildi = cv2.imwrite(str(onizleme_yolu), onizleme)

if not kaydedildi:
    raise RuntimeError("Ön izleme Drive klasörüne kaydedilemedi.")

print("Yeşil ön izleme kaydedildi:", onizleme_yolu)
# -------------------------------------------------
# 6. Koordinatları yazdır
# -------------------------------------------------

yukseklik, genislik = (
    orijinal_gorsel.shape[:2]
)

print("\nPiksel koordinatları:")
print(alt_yol_noktalari)

print("\nOransal koordinatlar:")

for x, y in alt_yol_noktalari:

    print(
        f"nokta({x / genislik:.4f}, "
        f"{y / yukseklik:.4f}),"
    )



---


10. AŞAMA — Alt Yol Maskesinin Video Üzerine Yerleştirilmesi

BU KOD;

videonun tamamında önceki bölümde seçtiğimiz alt yol bölgesini yeşil gösteren yeni bir video üretir.

In [ ]:
import cv2
import subprocess

from IPython.display import Video, display

#İşlenmiş karelerden geçici bir video oluşturur: /content/alt_yol_kontrol_gecici.mp4
gecici_video = (
    "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/"
    "alt_yol_kontrol_gecici.mp4"
)

sonuc_video = (
    "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/"
    "alt_yol_kontrol.mp4"
)


giris_video = cv2.VideoCapture(
    video_yolu
)

fps = giris_video.get(
    cv2.CAP_PROP_FPS
)

genislik = int(
    giris_video.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

yukseklik = int(
    giris_video.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)


# Görüntü ve video aynı boyutta olmalı
if (
    genislik != 1920
    or yukseklik != 1080
):
    raise RuntimeError(
        "Video çözünürlüğü 1920 × 1080 değil."
    )


video_yazici = cv2.VideoWriter(
    gecici_video,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (genislik, yukseklik)
)


while True:

    basarili, kare = giris_video.read()

    if not basarili:
        break

    katman = kare.copy()

    # Alt yolun içini yeşil boya
    cv2.fillPoly(
        katman,
        [alt_yol_noktalari],
        (0, 255, 0)
    )

    # Saydam katmanı görüntüyle birleştir
    kare = cv2.addWeighted(
        katman,
        0.30,
        kare,
        0.70,
        0
    )

    # Alt yolun dış sınırını çiz
    cv2.polylines(
        kare,
        [alt_yol_noktalari],
        True,
        (0, 255, 0),
        5
    )

    cv2.putText(
        kare,
        "ALT YOL",
        (640, 615),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        3
    )

    video_yazici.write(kare)


giris_video.release()
video_yazici.release()


# Tarayıcı uyumlu H.264 videosu oluştur
subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-loglevel",
        "error",
        "-i",
        gecici_video,
        "-vcodec",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        sonuc_video
    ],
    check=True
)


display(
    Video(
        sonuc_video,
        embed=True,
        width=700
    )
)



---


11. Aşama — Seçili Yol Bölgesindeki Araçları Tespit Etme

Bu kod videoyu kare kare inceler. YOLO araçları bulur, ByteTrack araçları kareler boyunca takip ederek kimlik numarası verir. Kod yalnızca araç kutusunun en az %10’u seçili yol bölgesiyle kesişen araçları gösterir.
- Yeşil alan: Seçilen yol bölgesini gösterir.
- Renkli kutular: Tespit edilen araç türlerini ve takip numaralarını gösterir.
- Kırmızı alan: Araç kutularının yol bölgesinde kalan kısımlarını gösterir.
- Bilgi paneli: Her karede yol içinde kalan araç kutularının piksel alanını, doluluk yüzdesini ve araç sayısını gösterir.

In [ ]:
import cv2
import numpy as np
import subprocess
from pathlib import Path
from IPython.display import Video, display

# Sonuçların kaydedileceği klasör
Path(
    "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit"
).mkdir(parents=True, exist_ok=True)

# Kaynak ham videoyu aç
giris_video = cv2.VideoCapture(
    "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/trafik_akisi.mp4"
)

if not giris_video.isOpened():
    raise RuntimeError("Video açılamadı. Dosya yolunu kontrol edin.")

fps = giris_video.get(cv2.CAP_PROP_FPS)
genislik = int(giris_video.get(cv2.CAP_PROP_FRAME_WIDTH))
yukseklik = int(giris_video.get(cv2.CAP_PROP_FRAME_HEIGHT))

if fps <= 0:
    fps = 25

# Yol koordinatlarını görüntü boyutuna uygun biçime dönüştür
yol_cokgeni = np.array(
    alt_yol_noktalari,
    dtype=np.int32
).reshape(-1, 1, 2)

# Seçili yol bölgesinin maskesini oluştur
yol_maskesi = np.zeros((yukseklik, genislik), dtype=np.uint8)
cv2.fillPoly(yol_maskesi, [yol_cokgeni], 255)

yol_alani = cv2.countNonZero(yol_maskesi)

if yol_alani == 0:
    giris_video.release()
    raise ValueError("Seçili yol alanı boş. Koordinatları kontrol edin.")

# Video ve yol maskesi aynı boyutta olmalı
if yol_maskesi.shape != (yukseklik, genislik):
    giris_video.release()
    raise ValueError("Video ile yol maskesinin boyutları eşleşmiyor.")

# Modeldeki araç sınıflarını bul
if isinstance(model.names, dict):
    sinif_adlari = {
        int(numara): str(ad).lower()
        for numara, ad in model.names.items()
    }
else:
    sinif_adlari = {
        numara: str(ad).lower()
        for numara, ad in enumerate(model.names)
    }

arac_adlari = {"car", "motorcycle", "motorbike", "bus", "truck"}

arac_sinif_numaralari = [
    numara
    for numara, ad in sinif_adlari.items()
    if ad in arac_adlari
]

if not arac_sinif_numaralari:
    giris_video.release()
    raise ValueError("Modelde araç sınıfları bulunamadı.")

# Araç türlerine göre kutu renkleri (BGR)
renkler = {
    "car": (255, 0, 0),
    "motorcycle": (0, 255, 255),
    "motorbike": (0, 255, 255),
    "bus": (255, 0, 255),
    "truck": (0, 0, 255)
}

# Geçici video dosyasını oluştur
video_yazici = cv2.VideoWriter(
    "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/alt_yol_arac_tespit_gecici.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (genislik, yukseklik)
)

if not video_yazici.isOpened():
    giris_video.release()
    raise RuntimeError("Video yazıcısı açılamadı.")

minimum_kesisim_orani = 0.10
islenen_kare = 0

while True:
    basarili, ham_kare = giris_video.read()

    if not basarili:
        break

    # Tespiti yeşil kaplama eklenmemiş ham karede yap
    tahminler = model.track(
        source=ham_kare,
        persist=True,
        tracker="bytetrack.yaml",
        classes=arac_sinif_numaralari,
        conf=0.20,
        imgsz=1280,
        verbose=False
    )

    sonuc = tahminler[0]
    arac_maskesi = np.zeros((yukseklik, genislik), dtype=np.uint8)
    tespitler = []

    if sonuc.boxes is not None and len(sonuc.boxes) > 0:
        kutular = sonuc.boxes.xyxy.cpu().numpy()
        siniflar = sonuc.boxes.cls.int().cpu().tolist()
        guvenler = sonuc.boxes.conf.cpu().tolist()

        if sonuc.boxes.id is not None:
            takip_idleri = sonuc.boxes.id.int().cpu().tolist()
        else:
            takip_idleri = [None] * len(kutular)

        for kutu, sinif_no, guven, takip_id in zip(
            kutular, siniflar, guvenler, takip_idleri
        ):
            x1, y1, x2, y2 = map(int, kutu)

            x1 = max(0, min(x1, genislik - 1))
            x2 = max(0, min(x2, genislik - 1))
            y1 = max(0, min(y1, yukseklik - 1))
            y2 = max(0, min(y2, yukseklik - 1))

            if x2 <= x1 or y2 <= y1:
                continue

            # Araç kutusunun yol alanıyla kesişme oranını hesapla
            yol_kesiti = yol_maskesi[y1:y2 + 1, x1:x2 + 1]
            kutu_alani = (x2 - x1 + 1) * (y2 - y1 + 1)
            yol_pikseli = cv2.countNonZero(yol_kesiti)

            if yol_pikseli / kutu_alani < minimum_kesisim_orani:
                continue

            cv2.rectangle(
                arac_maskesi,
                (x1, y1),
                (x2, y2),
                255,
                thickness=-1
            )

            sinif_adi = sinif_adlari[int(sinif_no)]
            tespitler.append(
                (x1, y1, x2, y2, sinif_adi, guven, takip_id)
            )

    # Araç kutularının yol içindeki birleşik alanını hesapla
    yol_icindeki_araclar = cv2.bitwise_and(
        arac_maskesi,
        yol_maskesi
    )

    arac_piksel_alani = cv2.countNonZero(yol_icindeki_araclar)
    doluluk = arac_piksel_alani / yol_alani * 100

    # Yol alanını yeşil, araç kutularının yol içindeki bölümlerini kırmızı göster
    kare = ham_kare.copy()
    yol_ici = yol_maskesi > 0

    yesil = np.array([0, 255, 0], dtype=np.float32)
    kare[yol_ici] = (
        kare[yol_ici].astype(np.float32) * 0.72 + yesil * 0.28
    ).astype(np.uint8)

    arac_bolge = yol_icindeki_araclar > 0
    kirmizi = np.array([0, 0, 255], dtype=np.float32)
    kare[arac_bolge] = (
        kare[arac_bolge].astype(np.float32) * 0.25 + kirmizi * 0.75
    ).astype(np.uint8)

    cv2.polylines(kare, [yol_cokgeni], True, (0, 255, 0), 4)

    # Araç kutularını ve takip numaralarını çiz
    for x1, y1, x2, y2, sinif_adi, guven, takip_id in tespitler:
        renk = renkler.get(sinif_adi, (255, 255, 255))

        cv2.rectangle(kare, (x1, y1), (x2, y2), renk, 3)

        if takip_id is None:
            etiket = f"{sinif_adi} %{guven * 100:.0f}"
        else:
            etiket = f"{sinif_adi} ID:{takip_id}"

        cv2.putText(
            kare,
            etiket,
            (x1, max(25, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            renk,
            2
        )

    # Karedeki ölçüm bilgileri
    cv2.rectangle(kare, (20, 20), (650, 115), (0, 0, 0), -1)
    cv2.putText(
        kare,
        f"Yol icindeki arac kutusu: {arac_piksel_alani} piksel",
        (35, 55),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )
    cv2.putText(
        kare,
        f"Doluluk: %{doluluk:.2f}   Arac: {len(tespitler)}",
        (35, 95),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    video_yazici.write(kare)
    islenen_kare += 1

giris_video.release()
video_yazici.release()

# H.264 biçiminde son videoyu oluştur
subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-loglevel",
        "error",
        "-i",
        "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/alt_yol_arac_tespit_gecici.mp4",
        "-vcodec",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/alt_yol_arac_tespit.mp4"
    ],
    check=True
)

print("İşlenen kare sayısı:", islenen_kare)
print("Sonuç videosu Drive'a kaydedildi.")

display(
    Video(
        "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/alt_yol_arac_tespit.mp4",
        embed=True,
        width=900
    )
)

12. Aşama — Kare Kare Yol Doluluk Oranını ve Yoğunluk Eşiklerini Hesaplama

Bu kod, videoyu kare kare analiz eder. Her karede seçilen yol bölgesindeki araç kutularını bulur ve araç kutularının yol içinde kalan alanını hesaplar. Böylece her kare için yol doluluk yüzdesi elde edilir.
1. Videoyu ve yol bölgesini hazırlar.
   Yol poligonu, video boyutlarında siyah-beyaz bir maske hâline getirilir. Maskedeki beyaz pikseller seçilen yol alanını gösterir.
2. Her karede araçları tespit eder.
   YOLO ve ByteTrack araçları bulur. Araç kutusunun en az %10’u yol bölgesine giriyorsa araç hesaba katılır.
3. Yol içindeki araç kutusu alanını sayar.
   Araç kutuları bir maskede birleştirilir. Kesişen kutuların ortak pikselleri bir kez sayılır. Hesaplanan alan, yol alanına bölünerek doluluk yüzdesi bulunur.
4. Hareketli ortalamayı ve eşikleri hesaplar.
   Yaklaşık bir saniyelik hareketli ortalama, kareler arasındaki ani değişimleri yumuşatır. Bu ortalamaların 33. ve 67. yüzdelik değerleri düşük/orta ve orta/yüksek eşikleri olarak yazdırılır.
5. Sonuçları kaydeder.
   Kare numarası, zaman, araç sayısı, araç kutusu alanı ve doluluk değerleri bir tabloda gösterilir ve CSV dosyasına kaydedilir.

In [ ]:
import cv2
import numpy as np
import pandas as pd

# Videoyu aç
giris_video = cv2.VideoCapture(
    "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/trafik_akisi.mp4"
)

if not giris_video.isOpened():
    raise RuntimeError("Video açılamadı. Dosya yolunu kontrol edin.")

fps = giris_video.get(cv2.CAP_PROP_FPS)
beklenen_kare = int(giris_video.get(cv2.CAP_PROP_FRAME_COUNT))
genislik = int(giris_video.get(cv2.CAP_PROP_FRAME_WIDTH))
yukseklik = int(giris_video.get(cv2.CAP_PROP_FRAME_HEIGHT))

if fps <= 0:
    fps = 25

print("Videonun bildirdiği kare sayısı:", beklenen_kare)
print("Video boyutu:", genislik, "x", yukseklik)
print("FPS:", fps)

# Yol poligonu ve maskesi
yol_cokgeni = np.array(
    alt_yol_noktalari,
    dtype=np.int32
).reshape(-1, 1, 2)

yol_maskesi = np.zeros((yukseklik, genislik), dtype=np.uint8)
cv2.fillPoly(yol_maskesi, [yol_cokgeni], 255)

yol_alani = cv2.countNonZero(yol_maskesi)

if yol_alani == 0:
    giris_video.release()
    raise ValueError("Seçilen yol alanı boş.")

# Modeldeki araç sınıfları
if isinstance(model.names, dict):
    sinif_adlari = {
        int(numara): str(ad).lower()
        for numara, ad in model.names.items()
    }
else:
    sinif_adlari = {
        numara: str(ad).lower()
        for numara, ad in enumerate(model.names)
    }

arac_adlari = {"car", "motorcycle", "motorbike", "bus", "truck"}

arac_sinif_numaralari = [
    numara
    for numara, ad in sinif_adlari.items()
    if ad in arac_adlari
]

if not arac_sinif_numaralari:
    giris_video.release()
    raise ValueError("Modelde araç sınıfları bulunamadı.")

minimum_kesisim_orani = 0.10
kare_sonuclari = []
kare_no = 0

print("Kareler analiz ediliyor...")

while True:
    basarili, kare = giris_video.read()

    if not basarili:
        break

    tahminler = model.track(
        source=kare,
        persist=True,
        tracker="bytetrack.yaml",
        classes=arac_sinif_numaralari,
        conf=0.20,
        imgsz=1280,
        verbose=False
    )

    sonuc = tahminler[0]
    arac_maskesi = np.zeros((yukseklik, genislik), dtype=np.uint8)
    yol_icindeki_arac_sayisi = 0

    if sonuc.boxes is not None and len(sonuc.boxes) > 0:
        kutular = sonuc.boxes.xyxy.cpu().numpy()
        siniflar = sonuc.boxes.cls.int().cpu().tolist()

        for kutu, sinif_no in zip(kutular, siniflar):
            x1, y1, x2, y2 = map(int, kutu)

            x1 = max(0, min(x1, genislik - 1))
            x2 = max(0, min(x2, genislik - 1))
            y1 = max(0, min(y1, yukseklik - 1))
            y2 = max(0, min(y2, yukseklik - 1))

            if x2 <= x1 or y2 <= y1:
                continue

            # Araç kutusunun yol içinde kalan oranı
            yol_kesiti = yol_maskesi[y1:y2 + 1, x1:x2 + 1]
            kutu_alani = (x2 - x1 + 1) * (y2 - y1 + 1)
            yol_pikseli = cv2.countNonZero(yol_kesiti)

            if yol_pikseli / kutu_alani < minimum_kesisim_orani:
                continue

            cv2.rectangle(
                arac_maskesi,
                (x1, y1),
                (x2, y2),
                255,
                thickness=-1
            )

            yol_icindeki_arac_sayisi += 1

    # Üst üste gelen kutuların ortak piksellerini bir kez say
    yol_icindeki_araclar = cv2.bitwise_and(
        arac_maskesi,
        yol_maskesi
    )

    arac_kutu_alani = cv2.countNonZero(yol_icindeki_araclar)
    doluluk_yuzdesi = arac_kutu_alani / yol_alani * 100

    kare_no += 1

    kare_sonuclari.append({
        "kare": kare_no,
        "zaman_saniye": (kare_no - 1) / fps,
        "yol_alani_piksel": yol_alani,
        "yol_icindeki_arac_sayisi": yol_icindeki_arac_sayisi,
        "arac_kutu_alani_piksel": arac_kutu_alani,
        "doluluk_yuzdesi": doluluk_yuzdesi
    })

giris_video.release()

# Sonuç tablosu
sonuc_tablosu = pd.DataFrame(kare_sonuclari)

if sonuc_tablosu.empty:
    raise RuntimeError("Videodan kare okunamadı.")

# Yaklaşık 1 saniyelik hareketli ortalama
pencere = max(1, round(fps))

sonuc_tablosu["doluluk_ortalamasi_1sn"] = (
    sonuc_tablosu["doluluk_yuzdesi"]
    .rolling(window=pencere, center=True, min_periods=1)
    .mean()
)

# Videodaki 1 saniyelik ortalama dolulukların 33. ve 67. yüzdelik eşikleri
dusuk_orta_esigi, orta_yuksek_esigi = np.percentile(
    sonuc_tablosu["doluluk_ortalamasi_1sn"],
    [33, 67]
)

print("\nHesaplanan kare sayısı:", len(sonuc_tablosu))
print("Seçili yol alanı:", yol_alani, "piksel")
print(
    "Ortalama doluluk:",
    f"%{sonuc_tablosu['doluluk_ortalamasi_1sn'].mean():.2f}"
)
print(f"Düşük / orta eşiği: %{dusuk_orta_esigi:.2f}")
print(f"Orta / yüksek eşiği: %{orta_yuksek_esigi:.2f}")

print("\nİlk 5 kare:")
display(sonuc_tablosu.head())

# Kare kare sonuçları Drive'a CSV olarak kaydet
sonuc_tablosu.to_csv(
    "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/yogunluk_kare_sonuclari.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nKare sonuçları CSV dosyasına kaydedildi.")

13. Aşama — Trafik Yoğunluğu Eşiklerinin Belirlenmesi


Bu aşamada videodaki her kare için hesaplanan 1 saniyelik hareketli doluluk ortalamasını kullanarak düşük, orta ve yüksek yoğunluk sınırlarını buluyoruz. Hareketli ortalama, kareler arasındaki ani değişimleri yumuşatır.

Mevcut CSV’de 250 kare bulunuyor. Bu verinin 33. ve 67. yüzdelik değerlerine göre eşikler:
- Düşük: %21,45’in altı
- Orta: %21,45 dâhil, %28,68’in altı
- Yüksek: %28,68 ve üzeri
Bu eşikler mevcut videoya ait deneysel değerlerdir. Videonun tamamında 260 kare varsa eksik 10 karenin sonuçları eklendikten sonra eşikleri yeniden hesaplamak gerekir. Sonraki aşamada bu sınıflar yol bölgesini yeşil, sarı ve kırmızı renkte göstermek için kullanılacaktır.

DİKKAT: BU KOD ÖNCEKİ AŞAMADA OLUŞTURULAN "yogunluk_kare_sonuclari.csv" DOSYASINDAN DEĞERLERİ ALIP YOL RENKLENDİRMESİ YAPAR.

In [ ]:
import cv2
import numpy as np
import pandas as pd
import subprocess
from pathlib import Path
from IPython.display import Video, display

# CSV sonuçlarını oku ve kare numarasına göre sırala
veri = pd.read_csv(
    "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/yogunluk_kare_sonuclari.csv"
).sort_values("kare").reset_index(drop=True)

# CSV'deki 1 saniyelik doluluk ortalamalarından eşikleri hesapla
dusuk_orta_esigi, orta_yuksek_esigi = np.percentile(
    veri["doluluk_ortalamasi_1sn"].dropna(),
    [33, 67]
)

print(f"Düşük / orta eşiği: %{dusuk_orta_esigi:.2f}")
print(f"Orta / yüksek eşiği: %{orta_yuksek_esigi:.2f}")
print("CSV'deki kare sayısı:", len(veri))

# Sonuç klasörünü hazırla
Path(
    "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit"
).mkdir(parents=True, exist_ok=True)

# Kaynak videoyu aç
giris_video = cv2.VideoCapture(
    "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/trafik_akisi.mp4"
)

if not giris_video.isOpened():
    raise RuntimeError("Video açılamadı. Dosya yolunu kontrol edin.")

fps = giris_video.get(cv2.CAP_PROP_FPS)
genislik = int(giris_video.get(cv2.CAP_PROP_FRAME_WIDTH))
yukseklik = int(giris_video.get(cv2.CAP_PROP_FRAME_HEIGHT))
video_kare_sayisi = int(giris_video.get(cv2.CAP_PROP_FRAME_COUNT))

if fps <= 0:
    fps = 25

# Yol poligonunu video ölçülerine uygun biçime getir
yol_cokgeni = np.array(
    alt_yol_noktalari,
    dtype=np.int32
).reshape(-1, 1, 2)

# Video ile CSV'de ölçülen kareleri eşleştir
islenebilecek_kare = min(len(veri), video_kare_sayisi)

if len(veri) != video_kare_sayisi:
    print(
        f"Uyarı: CSV'de {len(veri)}, videoda {video_kare_sayisi} kare var. "
        f"Yalnızca eşleşen {islenebilecek_kare} kare renklendirilecek."
    )

# Yoğunluk adları ve renkleri
yogunluk_adi = {
    "DUSUK": "DUSUK",
    "ORTA": "ORTA",
    "YUKSEK": "YUKSEK"
}

yogunluk_rengi = {
    "DUSUK": (0, 255, 0),      # Yeşil
    "ORTA": (0, 255, 255),     # Sarı
    "YUKSEK": (0, 0, 255)      # Kırmızı
}

# Geçici videoyu oluştur
video_yazici = cv2.VideoWriter(
    "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/yol_yogunluk_gecici.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (genislik, yukseklik)
)

if not video_yazici.isOpened():
    giris_video.release()
    raise RuntimeError("Sonuç videosu oluşturulamadı.")

for sira in range(islenebilecek_kare):
    basarili, kare = giris_video.read()

    if not basarili:
        break

    doluluk = float(veri.loc[sira, "doluluk_ortalamasi_1sn"])

    if doluluk < dusuk_orta_esigi:
        yogunluk = "DUSUK"
    elif doluluk < orta_yuksek_esigi:
        yogunluk = "ORTA"
    else:
        yogunluk = "YUKSEK"

    renk = yogunluk_rengi[yogunluk]

    # Yol bölgesini yoğunluk rengine göre yarı saydam boya
    katman = kare.copy()
    cv2.fillPoly(katman, [yol_cokgeni], renk)
    kare = cv2.addWeighted(katman, 0.30, kare, 0.70, 0)

    # Yol sınırını çiz
    cv2.polylines(
        kare,
        [yol_cokgeni],
        True,
        renk,
        4
    )

    # Kare numarası ve doluluk bilgisi
    cv2.rectangle(kare, (20, 20), (560, 105), (0, 0, 0), -1)

    cv2.putText(
        kare,
        f"TRAFIK YOGUNLUGU: {yogunluk_adi[yogunluk]}",
        (35, 55),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.75,
        renk,
        2
    )

    cv2.putText(
        kare,
        f"1 sn ortalama doluluk: %{doluluk:.2f}",
        (35, 90),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2
    )

    video_yazici.write(kare)

giris_video.release()
video_yazici.release()

# H.264 biçiminde kaydet
subprocess.run(
    [
        "ffmpeg", "-y", "-loglevel", "error",
        "-i",
        "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/yol_yogunluk_gecici.mp4",
        "-vcodec", "libx264",
        "-pix_fmt", "yuv420p",
        "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/yol_yogunluk_renkli.mp4"
    ],
    check=True
)

print("Renklendirilen kare sayısı:", islenebilecek_kare)
print("Sonuç videosu Drive'a kaydedildi.")

display(
    Video(
        "/content/drive/MyDrive/YOLO_Projeleri/4_Yol_Video_Tespit/yol_yogunluk_renkli.mp4",
        embed=True,
        width=900
    )
)



---


14. Aşama - Sağ, Sol, Üst Yolların Paintte İşaretlenmesi ve Video Üzerine Eklenmesi

DİKKAT:

9 NUMARALI AŞAMADAKİ BASAMAKLARI TAKİP EDEREK SAĞ, SOL VE ÜST İÇİN DE YAPARAK BU AŞAMAYI KENDİNİZ YAPINIZ



---


15. AŞAMA — Dört Yol Bölgesinin Birlikte Kontrol Edilmesi

| Yol doluluk oranı             | Sınıf  | Renk       |
| ----------------------------- | ------ | ---------- |
| **%3 ve altı**                | Düşük  | 🟢 Yeşil   |
| **%3’ten büyük, %10 ve altı** | Orta   | 🟡 Sarı    |
| **%10’dan büyük**             | Yüksek | 🔴 Kırmızı |


In [ ]:
# ============================================================
# TÜM YOLLAR İÇİN TRAFİK YOĞUNLUĞU ANALİZİ
# YOLO11s + ByteTrack + Yol Bazlı Renkli Gösterim
# ============================================================

import cv2
import time
import subprocess
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from ultralytics import YOLO
from IPython.display import Video, display
from google.colab import files


# ------------------------------------------------------------
# 1. AYARLAR
# ------------------------------------------------------------

# Daha önce video_yolu değişkeni tanımlandıysa onu kullanır.
# Tanımlanmadıysa aşağıdaki yolu kullanır.
video_yolu = globals().get("video_yolu", "/content/vid.mp4")

model_yolu = "yolo11s.pt"

gecici_video_yolu = "/content/tum_yollar_gecici.mp4"
sonuc_video_yolu = "/content/tum_yollar_trafik_yogunlugu.mp4"

# COCO araç sınıfları:
# 2 = car, 3 = motorcycle, 5 = bus, 7 = truck
arac_siniflari = [2, 3, 5, 7]

guven_esigi = 0.15
goruntu_boyutu = 960

if not Path(video_yolu).exists():
    raise FileNotFoundError(
        f"Video bulunamadı: {video_yolu}\n"
        "video_yolu değişkenini doğru dosya yoluyla değiştirin."
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU bulunamadı. Colab'da Çalışma zamanı > "
        "Çalışma zamanı türünü değiştir > T4 GPU seçin."
    )

print("Kullanılan GPU:", torch.cuda.get_device_name(0))
print("Video:", video_yolu)


# ------------------------------------------------------------
# 2. VİDEO BİLGİLERİNİ OKUMA
# ------------------------------------------------------------

video = cv2.VideoCapture(video_yolu)

fps = video.get(cv2.CAP_PROP_FPS)
genislik = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
yukseklik = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
toplam_kare = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

video.release()

if fps <= 0 or genislik <= 0 or yukseklik <= 0:
    raise RuntimeError("Video bilgileri okunamadı.")

print("\nVideo çözünürlüğü:", genislik, "x", yukseklik)
print("FPS:", round(fps, 2))
print("Toplam kare:", toplam_kare)


# ------------------------------------------------------------
# 3. YOL BÖLGELERİNİ TANIMLAMA
# ------------------------------------------------------------

def nokta(x_orani, y_orani):
    """
    Oransal koordinatları piksel koordinatına dönüştürür.
    Böylece farklı çözünürlüklerde aynı yol geometrisi korunur.
    """
    return [
        int(x_orani * genislik),
        int(y_orani * yukseklik)
    ]


yol_bolgeleri = {
    # Paint çizgilerinden çıkarılan kesin yol sınırları
    "SOL YOL": np.array([
        nokta(0.3995, 0.4241),
        nokta(0.2984, 0.5787),
        nokta(0.0000, 0.5648),
        nokta(0.0000, 0.4315)
    ], dtype=np.int32),

    "SAG YOL": np.array([
        nokta(0.5151, 0.5880),
        nokta(0.5510, 0.4287),
        nokta(0.9958, 0.4694),
        nokta(0.9943, 0.6481)
    ], dtype=np.int32),

    "UST YOL": np.array([
        nokta(0.7359, 0.0000),
        nokta(0.5536, 0.4315),
        nokta(0.4224, 0.4287),
        nokta(0.6542, 0.0000)
    ], dtype=np.int32),

    "ALT YOL": np.array([
        nokta(0.4625, 0.5926),
        nokta(0.2953, 0.9991),
        nokta(0.1255, 0.9991),
        nokta(0.3292, 0.5824)
    ], dtype=np.int32)
}


# Her yol için ayrı maske oluştur
yol_maskeleri = {}
yol_alanlari = {}

for yol_adi, cokgen in yol_bolgeleri.items():

    maske = np.zeros(
        (yukseklik, genislik),
        dtype=np.uint8
    )

    cv2.fillPoly(
        maske,
        [cokgen],
        255
    )

    yol_maskeleri[yol_adi] = maske
    yol_alanlari[yol_adi] = cv2.countNonZero(maske)


# ------------------------------------------------------------
# 4. YOLO11s + BYTETRACK İLE ARAÇ TAKİBİ
# ------------------------------------------------------------

print("\nYOLO11s ve ByteTrack analizi başladı...")

model = YOLO(model_yolu)

gpu_baslangic = time.perf_counter()

takip_sonuclari = model.track(
    source=video_yolu,
    tracker="bytetrack.yaml",
    persist=True,
    classes=arac_siniflari,
    conf=guven_esigi,
    imgsz=goruntu_boyutu,
    device=0,
    quantize=16,
    stream=True,
    verbose=False
)

kare_kayitlari = []
doluluk_kayitlari = []
benzersiz_idler = set()


for kare_numarasi, sonuc in enumerate(takip_sonuclari):

    # Araç kutuları
    if sonuc.boxes is not None and len(sonuc.boxes) > 0:

        kutular = sonuc.boxes.xyxy.cpu().numpy().astype(int)
        siniflar = sonuc.boxes.cls.cpu().numpy().astype(int)

        if sonuc.boxes.id is not None:
            takip_idleri = sonuc.boxes.id.cpu().numpy().astype(int)
        else:
            takip_idleri = np.full(len(kutular), -1)

    else:
        kutular = np.empty((0, 4), dtype=int)
        siniflar = np.empty(0, dtype=int)
        takip_idleri = np.empty(0, dtype=int)

    for takip_id in takip_idleri:
        if takip_id >= 0:
            benzersiz_idler.add(int(takip_id))

    # Bu karedeki tüm araç kutularını maskele
    arac_maskesi = np.zeros(
        (yukseklik, genislik),
        dtype=np.uint8
    )

    for kutu in kutular:

        x1, y1, x2, y2 = kutu

        x1 = np.clip(x1, 0, genislik - 1)
        x2 = np.clip(x2, 0, genislik - 1)
        y1 = np.clip(y1, 0, yukseklik - 1)
        y2 = np.clip(y2, 0, yukseklik - 1)

        cv2.rectangle(
            arac_maskesi,
            (x1, y1),
            (x2, y2),
            255,
            -1
        )

    kare_doluluklari = {}

    # Her yolun araçlarla kaplanan alanını ayrı hesapla
    for yol_adi, yol_maskesi in yol_maskeleri.items():

        yol_uzerindeki_arac_alani = cv2.bitwise_and(
            arac_maskesi,
            yol_maskesi
        )

        kaplanan_alan = cv2.countNonZero(
            yol_uzerindeki_arac_alani
        )

        doluluk_yuzdesi = (
            kaplanan_alan /
            yol_alanlari[yol_adi]
        ) * 100

        kare_doluluklari[yol_adi] = doluluk_yuzdesi

    kare_kayitlari.append({
        "kare": kare_numarasi,
        "kutular": kutular,
        "siniflar": siniflar,
        "takip_idleri": takip_idleri,
        "aktif_arac": len(kutular)
    })

    doluluk_kayitlari.append(kare_doluluklari)


gpu_suresi = time.perf_counter() - gpu_baslangic

print("Araç analizi tamamlandı.")
print("GPU analiz süresi:", round(gpu_suresi, 2), "saniye")
print("Benzersiz takip ID sayısı:", len(benzersiz_idler))


# ------------------------------------------------------------
# 5. DOLULUK VERİ TABLOSUNU OLUŞTURMA
# ------------------------------------------------------------

grafik_verisi = pd.DataFrame(doluluk_kayitlari)

grafik_verisi.insert(
    0,
    "zaman_saniye",
    np.arange(len(grafik_verisi)) / fps
)

# Yaklaşık bir saniyelik hareketli ortalama
pencere_boyutu = max(1, int(round(fps)))

for yol_adi in yol_bolgeleri:

    grafik_verisi[yol_adi + "_ORT"] = (
        grafik_verisi[yol_adi]
        .rolling(
            window=pencere_boyutu,
            center=True,
            min_periods=1
        )
        .mean()
    )


# ------------------------------------------------------------
# 6. BÜTÜN YOLLAR İÇİN ORTAK YOĞUNLUK EŞİKLERİ
# ------------------------------------------------------------

# Yol başına quantile kullanmak, çoğunlukla boş bir yolu
# tek araçla bile YÜKSEK gösterebildiği için kaldırılmıştır.
# Bütün yollar aynı fiziksel doluluk ölçeğinde değerlendirilir.
# Mevcut videodaki gözlemlere göre genişletilen aralık:
# %3 ve altı     -> DÜŞÜK
# %3 - %10 arası -> ORTA
# %10 üzeri      -> YÜKSEK
ortak_alt_esik = 3.00
ortak_ust_esik = 9.00

yol_esikleri = {
    yol_adi: {
        "alt": ortak_alt_esik,
        "ust": ortak_ust_esik
    }
    for yol_adi in yol_bolgeleri
}

print("\nBütün yollar için kullanılan ortak eşikler:")
print(f"DÜŞÜK : doluluk <= %{ortak_alt_esik:.2f}")
print(
    f"ORTA  : %{ortak_alt_esik:.2f} ile "
    f"%{ortak_ust_esik:.2f} arası"
)
print(f"YÜKSEK: doluluk > %{ortak_ust_esik:.2f}")


def yogunluk_sinifi(deger, alt_esik, ust_esik):

    # Yol doluluk değeri bütün video boyunca neredeyse
    # değişmiyorsa yanlış sınıflandırmayı engeller.
    if abs(ust_esik - alt_esik) < 0.001:
        return "DUSUK"

    if deger <= alt_esik:
        return "DUSUK"

    elif deger <= ust_esik:
        return "ORTA"

    else:
        return "YUKSEK"


# OpenCV renkleri BGR sırasındadır
renk_haritasi = {
    "DUSUK":  (0, 255, 0),     # Yeşil
    "ORTA":   (0, 255, 255),   # Sarı
    "YUKSEK": (0, 0, 255)      # Kırmızı
}


# ------------------------------------------------------------
# 7. RENKLİ SONUÇ VİDEOSUNU CPU İLE OLUŞTURMA
# ------------------------------------------------------------

print("\nRenkli sonuç videosu oluşturuluyor...")

cpu_baslangic = time.perf_counter()

giris_video = cv2.VideoCapture(video_yolu)

video_yazici = cv2.VideoWriter(
    gecici_video_yolu,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (genislik, yukseklik)
)

kare_numarasi = 0

while True:

    basarili, kare = giris_video.read()

    if not basarili:
        break

    if kare_numarasi >= len(kare_kayitlari):
        break

    kayit = kare_kayitlari[kare_numarasi]
    renkli_katman = kare.copy()

    # Her yolu kendi yoğunluğuna göre renklendir
    for yol_adi, cokgen in yol_bolgeleri.items():

        doluluk = grafik_verisi.loc[
            kare_numarasi,
            yol_adi + "_ORT"
        ]

        alt_esik = yol_esikleri[yol_adi]["alt"]
        ust_esik = yol_esikleri[yol_adi]["ust"]

        sinif = yogunluk_sinifi(
            doluluk,
            alt_esik,
            ust_esik
        )

        renk = renk_haritasi[sinif]

        cv2.fillPoly(
            renkli_katman,
            [cokgen],
            renk
        )

    # Saydam renkli katmanı gerçek kareyle birleştir
    kare = cv2.addWeighted(
        renkli_katman,
        0.28,
        kare,
        0.72,
        0
    )

    # Yol sınırlarını ve bilgilerini çiz
    yazi_y = 35

    for yol_adi, cokgen in yol_bolgeleri.items():

        doluluk = grafik_verisi.loc[
            kare_numarasi,
            yol_adi + "_ORT"
        ]

        alt_esik = yol_esikleri[yol_adi]["alt"]
        ust_esik = yol_esikleri[yol_adi]["ust"]

        sinif = yogunluk_sinifi(
            doluluk,
            alt_esik,
            ust_esik
        )

        renk = renk_haritasi[sinif]

        cv2.polylines(
            kare,
            [cokgen],
            True,
            renk,
            3
        )

        bilgi = (
            f"{yol_adi}: {sinif} - %{doluluk:.2f}"
        )

        # Yazının görünmesi için siyah arka plan
        cv2.rectangle(
            kare,
            (10, yazi_y - 24),
            (470, yazi_y + 7),
            (0, 0, 0),
            -1
        )

        cv2.putText(
            kare,
            bilgi,
            (18, yazi_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            renk,
            2
        )

        yazi_y += 38

    # Araç kutularını ve ByteTrack numaralarını çiz
    for kutu, takip_id, sinif_no in zip(
        kayit["kutular"],
        kayit["takip_idleri"],
        kayit["siniflar"]
    ):

        x1, y1, x2, y2 = map(int, kutu)

        cv2.rectangle(
            kare,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            2
        )

        if takip_id >= 0:
            arac_yazisi = f"ID:{takip_id}"
        else:
            arac_yazisi = "ARAC"

        cv2.putText(
            kare,
            arac_yazisi,
            (x1, max(20, y1 - 7)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 255),
            2
        )

    # Genel bilgi
    cv2.putText(
        kare,
        f"Aktif arac: {kayit['aktif_arac']}",
        (genislik - 260, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    video_yazici.write(kare)

    kare_numarasi += 1


giris_video.release()
video_yazici.release()

cpu_video_suresi = time.perf_counter() - cpu_baslangic

print(
    "CPU ile renkli video oluşturma süresi:",
    round(cpu_video_suresi, 2),
    "saniye"
)


# ------------------------------------------------------------
# 8. VİDEOYU H.264 FORMATINA DÖNÜŞTÜRME
# ------------------------------------------------------------

ffmpeg_baslangic = time.perf_counter()

subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i", gecici_video_yolu,
        "-vcodec", "libx264",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        sonuc_video_yolu
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=True
)

ffmpeg_suresi = time.perf_counter() - ffmpeg_baslangic


# ------------------------------------------------------------
# 9. SÜRE VE SONUÇ BİLGİLERİ
# ------------------------------------------------------------

toplam_sure = (
    gpu_suresi +
    cpu_video_suresi +
    ffmpeg_suresi
)

print("\n========== İŞLEM SÜRELERİ ==========")
print("YOLO + ByteTrack analiz süresi :", round(gpu_suresi, 2), "sn")
print("CPU görüntü oluşturma süresi   :", round(cpu_video_suresi, 2), "sn")
print("FFmpeg dönüştürme süresi       :", round(ffmpeg_suresi, 2), "sn")
print("Toplam işlem süresi            :", round(toplam_sure, 2), "sn")
print("Sonuç videosu                  :", sonuc_video_yolu)


# ------------------------------------------------------------
# 10. SONUÇ VİDEOSUNU GÖSTERME
# ------------------------------------------------------------

display(
    Video(
        sonuc_video_yolu,
        embed=True,
        width=900
    )
)